# Projeto ML - Detecção de DDoS em SDN

Este notebook contém todo o código do diretório `ml/`, unificado seguindo o guia de boas práticas.


## Parte 1: Pipeline de Classificação Binária


### Arquivo: `ml/config.py`


In [ ]:
"""
Configurações globais do pipeline de detecção de DDoS.

Centraliza todas as constantes — caminhos, hiperparâmetros, seeds e limiares —
de forma que qualquer alteração afete apenas este módulo (SRP).
"""

from pathlib import Path

# ── Reprodutibilidade ──────────────────────────────────────────────────────────
RANDOM_STATE: int = 42

# ── Caminhos ───────────────────────────────────────────────────────────────────
PROJECT_ROOT  = Path(__file__).resolve().parent.parent
DATASET_PATH  = PROJECT_ROOT / "dataset" / "insdn8_ddos_binary_0n1d.csv"
MODELS_DIR    = PROJECT_ROOT / "models"
OUTPUTS_DIR   = PROJECT_ROOT / "outputs"

# ── Coluna alvo ────────────────────────────────────────────────────────────────
TARGET_COL: str = "Label"

# ── Split ──────────────────────────────────────────────────────────────────────
# 70% treino / 30% teste — padrão do curso (Thaís Gaudencio, UFPB/LUMO)
TEST_SIZE: float = 0.30

# ── Pré-processamento ──────────────────────────────────────────────────────────
# Limiar de variância: features com variância <= threshold são removidas
VARIANCE_THRESHOLD: float = 0.0

# Estratégia de imputação: median é mais robusto para features assimétricas de rede
IMPUTER_STRATEGY: str = "median"

# ── Seleção de features (SHAP) ─────────────────────────────────────────────────
# Número máximo de instâncias para calcular SHAP (caro computacionalmente)
SHAP_SAMPLE_SIZE: int = 10_000
# Número de features a selecionar. None = manter todas (para datasets já enxutos)
N_FEATURES_TO_SELECT: int | None = None   # insdn8 já tem apenas 8 features

# ── Arquitetura MLP ────────────────────────────────────────────────────────────
# Conforme Mehmood et al. (PLoS ONE, 2025) — arquitetura baseline
MLP_HIDDEN_LAYERS: tuple = (128, 64)
MLP_ACTIVATION:    str   = "relu"
MLP_SOLVER:        str   = "adam"
MLP_ALPHA:         float = 0.0001      # regularização L2
MLP_MAX_ITER:      int   = 200
MLP_LEARNING_RATE: str   = "adaptive"  # reduz taxa se estagna
MLP_EARLY_STOP:    bool  = True
MLP_VAL_FRACTION:  float = 0.1         # 10% do treino como validação interna
MLP_N_ITER_NO_CHG: int   = 10          # ciclos sem melhora para parar

# ── Validação cruzada ──────────────────────────────────────────────────────────
CV_N_SPLITS: int = 5
CV_SCORING:  str = "f1"  # F1 equilibra precisão e recall — melhor para DDoS

# ── Hyperparameter Tuning ──────────────────────────────────────────────────────
TUNING_N_ITER: int = 30  # combinações a testar no RandomizedSearchCV
TUNING_PARAM_DISTRIBUTIONS: dict = {
    "hidden_layer_sizes": [
        (128, 64),        # baseline do artigo
        (256, 128),
        (128, 64, 32),
        (256, 128, 64),
        (512, 256, 128),
    ],
    "alpha":               [0.0001, 0.001, 0.01, 0.1],
    "learning_rate_init":  [0.001, 0.01, 0.0001],
    "learning_rate":       ["constant", "adaptive", "invscaling"],
    "max_iter":            [200, 300, 500],
}

# ── Métricas de referência (artigo Mehmood et al., 2025) ───────────────────────
PAPER_METRICS: dict = {
    "accuracy":  99.98,
    "precision": 99.99,
    "recall":    99.97,
    "f1":        99.98,
}


### Arquivo: `ml/utils/learning_curve_plotter.py`


In [ ]:
"""
Curvas de aprendizado e diagnóstico de overfit.

Gera três tipos de visualização para investigar overfitting:

  1. plot_learning_curve  — Acurácia/F1 treino vs. validação por tamanho de treino.
     Diagnóstico: se treino >> validação em todos os pontos → overfit.

  2. plot_train_val_gap   — Gap entre score de treino e validação ao longo das épocas
     (para MLP). Indica quando o modelo começa a memorizar.

  3. plot_cv_scores       — Distribuição dos scores dos folds do CV.
     Alta variância entre folds → instabilidade, possível overfit.

  4. plot_overfit_dashboard — Dashboard completo com os três gráficos.

Uso standalone (CLI):
    python -m ml.utils.learning_curve_plotter \\
        --model-path models_triclass/rf_triclass.joblib \\
        --data-dir   dataset/InSDN_DatasetCSV/

Uso como módulo:
    plotter = LearningCurvePlotter(model, X_train, y_train)
    plotter.plot_learning_curve()
    plotter.plot_cv_scores(cv_results_dict)
    plotter.plot_overfit_dashboard(mlp_model)
"""

from __future__ import annotations

from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from sklearn.model_selection import StratifiedKFold, learning_curve

# ── Diretório padrão para outputs ─────────────────────────────────────────────
try:
except ImportError:
    _OUTPUTS_DIR = Path("outputs_triclass")


class LearningCurvePlotter:
    """
    Gerador de curvas de aprendizado para diagnóstico de overfit.

    Parâmetros
    ----------
    model     : estimador sklearn (RF, MLP Pipeline, etc.)
    X_train   : features de treino (pós-VT, pré-SMOTE para diagnóstico real)
    y_train   : labels de treino
    scoring   : métrica de avaliação ('f1_macro' recomendado para triclasse)
    cv        : número de folds ou objeto CV
    output_dir: diretório para salvar os PNGs
    """

    def __init__(
        self,
        model: Any,
        X_train: np.ndarray,
        y_train: np.ndarray,
        scoring: str = "f1_macro",
        cv: int = 5,
        output_dir: Path | str = _OUTPUTS_DIR,
        random_state: int = 42,
    ) -> None:
        self._model       = model
        self._X           = np.array(X_train)
        self._y           = np.array(y_train)
        self._scoring     = scoring
        self._cv          = StratifiedKFold(
            n_splits=cv, shuffle=True, random_state=random_state
        )
        self._out         = Path(output_dir)
        self._out.mkdir(parents=True, exist_ok=True)
        self._random_state = random_state

    # ── API pública ────────────────────────────────────────────────────────────

    def plot_learning_curve(
        self,
        train_sizes: np.ndarray | None = None,
        save: bool = True,
        title: str = "Curva de Aprendizado",
    ) -> dict[str, np.ndarray]:
        """
        Plota score de treino vs. validação para diferentes tamanhos de treino.

        Como interpretar:
          - Treino alto, Validação baixa → overfit (modelo memoriza).
          - Treino baixo, Validação baixa → underfit (modelo não aprendeu).
          - Treino ≈ Validação, ambos altos → generalização adequada.
          - Gap decresce com mais dados → mais dados ajudariam.
          - Gap estável com mais dados → mudança de modelo necessária.

        Returns
        -------
        dict com 'train_sizes', 'train_scores_mean', 'val_scores_mean'
        """
        if train_sizes is None:
            train_sizes = np.linspace(0.10, 1.0, 10)

        train_sizes_abs, train_scores, val_scores = learning_curve(
            self._model,
            self._X, self._y,
            train_sizes=train_sizes,
            cv=self._cv,
            scoring=self._scoring,
            n_jobs=-1,
            shuffle=True,
            random_state=self._random_state,
        )

        train_mean = train_scores.mean(axis=1)
        train_std  = train_scores.std(axis=1)
        val_mean   = val_scores.mean(axis=1)
        val_std    = val_scores.std(axis=1)
        gap        = train_mean - val_mean

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # ── Plot 1: curvas treino e validação ──────────────────────────────────
        ax = axes[0]
        ax.plot(train_sizes_abs, train_mean, "o-", color="steelblue",
                label=f"Treino ({self._scoring})")
        ax.fill_between(train_sizes_abs,
                        train_mean - train_std,
                        train_mean + train_std,
                        alpha=0.15, color="steelblue")
        ax.plot(train_sizes_abs, val_mean, "s--", color="darkorange",
                label=f"Validação CV ({self._scoring})")
        ax.fill_between(train_sizes_abs,
                        val_mean - val_std,
                        val_mean + val_std,
                        alpha=0.15, color="darkorange")

        ax.set_xlabel("Amostras de Treino")
        ax.set_ylabel(f"Score ({self._scoring})")
        ax.set_title(f"{title}\nTreino vs. Validação")
        ax.legend(loc="lower right")
        ax.set_ylim(max(0, min(val_mean) - 0.05), 1.02)
        ax.grid(alpha=0.3)

        # Anotar gap no último ponto
        last_gap = gap[-1]
        ax.annotate(
            f"Gap final: {last_gap:.4f}",
            xy=(train_sizes_abs[-1], (train_mean[-1] + val_mean[-1]) / 2),
            xytext=(-80, 0),
            textcoords="offset points",
            arrowprops=dict(arrowstyle="->", color="gray"),
            fontsize=9, color="crimson",
        )

        # ── Plot 2: gap ao longo dos tamanhos de treino ─────────────────────
        ax2 = axes[1]
        ax2.plot(train_sizes_abs, gap, "^-", color="crimson", label="Gap (Treino − Validação)")
        ax2.fill_between(train_sizes_abs, 0, gap, alpha=0.15, color="crimson")
        ax2.axhline(0.05, linestyle="--", color="gray", linewidth=0.8,
                    label="Threshold 0.05 (referência)")
        ax2.set_xlabel("Amostras de Treino")
        ax2.set_ylabel("Gap de Score")
        ax2.set_title("Gap Treino − Validação\n(overfit > 0.05 é preocupante)")
        ax2.legend()
        ax2.set_ylim(-0.02, max(gap) + 0.05)
        ax2.grid(alpha=0.3)

        # Diagnóstico automático
        overfit_flag = last_gap > 0.05
        underfit_flag = val_mean[-1] < 0.70
        diag = []
        if overfit_flag:
            diag.append("OVERFIT detectado (gap > 0.05 com treino completo)")
        if underfit_flag:
            diag.append("UNDERFIT suspeito (val < 0.70)")
        if not diag:
            diag.append("Sem sinais claros de overfit/underfit")

        fig.suptitle(
            f"{title}\nDiagnóstico: {' | '.join(diag)}",
            fontsize=11, fontweight="bold",
            color="crimson" if overfit_flag else "darkgreen",
        )

        plt.tight_layout()
        if save:
            slug = title.lower().replace(" ", "_").replace("/", "_")
            path = self._out / f"learning_curve_{slug}.png"
            plt.savefig(path, dpi=150, bbox_inches="tight")
            print(f"[LearningCurvePlotter] Salvo → {path.name}")
        plt.show()
        plt.close()

        return {
            "train_sizes":      train_sizes_abs,
            "train_scores_mean": train_mean,
            "val_scores_mean":   val_mean,
            "gap":               gap,
            "overfit_flag":      overfit_flag,
        }

    def plot_cv_scores(
        self,
        cv_results: dict[str, tuple[float, float]],
        label: str = "Modelo",
        save: bool = True,
    ) -> None:
        """
        Boxplot / barras dos scores por fold do CV.

        Alta variância entre folds → modelo instável.
        Variância baixa + score alto → boa generalização.

        Parameters
        ----------
        cv_results : dict retornado por TriclassTrainer.cross_validate_*
                     formato: {'f1_macro': (mean, std), 'accuracy': (mean, std)}
        """
        metrics  = list(cv_results.keys())
        means    = [cv_results[m][0] for m in metrics]
        stds     = [cv_results[m][1] for m in metrics]

        fig, ax = plt.subplots(figsize=(8, 5))
        x = np.arange(len(metrics))
        bars = ax.bar(x, means, yerr=stds, capsize=6, color="steelblue",
                      alpha=0.8, ecolor="crimson")

        for bar, mean, std in zip(bars, means, stds):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + std + 0.005,
                f"{mean:.4f}\n±{std:.4f}",
                ha="center", va="bottom", fontsize=9,
            )

        ax.set_xticks(x)
        ax.set_xticklabels(metrics, rotation=20, ha="right")
        ax.set_ylabel("Score (CV médio)")
        ax.set_ylim(max(0, min(means) - 3 * max(stds) - 0.05), 1.05)
        ax.set_title(f"Scores de Validação Cruzada — {label}\n"
                     f"(barra de erro = ±1 desvio padrão entre folds)")
        ax.grid(axis="y", alpha=0.3)

        # Diagnóstico de instabilidade
        max_std = max(stds) if stds else 0
        if max_std > 0.05:
            ax.text(0.98, 0.02,
                    f"⚠ Instabilidade: std máx = {max_std:.4f}",
                    transform=ax.transAxes, ha="right", va="bottom",
                    fontsize=9, color="crimson",
                    bbox=dict(boxstyle="round", facecolor="lightyellow"))

        plt.tight_layout()
        if save:
            slug = label.lower().replace(" ", "_")
            path = self._out / f"cv_scores_{slug}.png"
            plt.savefig(path, dpi=150, bbox_inches="tight")
            print(f"[LearningCurvePlotter] CV scores salvo → {path.name}")
        plt.show()
        plt.close()

    def plot_mlp_loss_gap(
        self,
        mlp_pipeline,
        label: str = "MLP",
        save: bool = True,
    ) -> None:
        """
        Plota loss de treino vs. score de validação interna do MLP por época.

        Requer MLP com early_stopping=True (gera validation_scores_).

        Como interpretar:
          - Loss treino cai mas val_score estagna → overfit começa.
          - Ponto de cruzamento val_score / loss → quando parar (early_stop).
          - Gap crescente após cruzamento → overfitting progressivo.
        """
        mlp = (
            mlp_pipeline.named_steps.get("mlp", None)
            if hasattr(mlp_pipeline, "named_steps")
            else mlp_pipeline
        )

        if mlp is None or not hasattr(mlp, "loss_curve_"):
            print("[LearningCurvePlotter] MLP sem loss_curve_ — treinar antes de plotar.")
            return

        loss_curve = mlp.loss_curve_
        val_scores = getattr(mlp, "validation_scores_", None)

        fig, ax1 = plt.subplots(figsize=(11, 5))
        epochs = np.arange(1, len(loss_curve) + 1)

        ax1.plot(epochs, loss_curve, color="steelblue", label="Loss de Treino", linewidth=1.5)
        ax1.set_xlabel("Épocas")
        ax1.set_ylabel("Loss de Treino", color="steelblue")
        ax1.tick_params(axis="y", labelcolor="steelblue")

        if val_scores is not None and len(val_scores) > 0:
            ax2 = ax1.twinx()
            # Alinhar val_scores ao mesmo comprimento do loss_curve
            v_epochs = np.linspace(1, len(loss_curve), len(val_scores))
            ax2.plot(v_epochs, val_scores, color="darkorange",
                     linestyle="--", label="Score Validação Interna", linewidth=1.5)
            ax2.set_ylabel("Score Validação", color="darkorange")
            ax2.tick_params(axis="y", labelcolor="darkorange")

            # Detectar ponto de overfitting: val_score começa a cair
            if len(val_scores) > 3:
                best_val_epoch = int(np.argmax(val_scores))
                best_epoch_abs = int(v_epochs[best_val_epoch])
                ax1.axvline(best_epoch_abs, color="green", linestyle=":",
                            linewidth=1.5, label=f"Melhor val (época {best_epoch_abs})")

            lines1, labels1 = ax1.get_legend_handles_labels()
            lines2, labels2 = ax2.get_legend_handles_labels()
            ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
        else:
            ax1.legend(loc="upper right")

        ax1.set_title(
            f"Convergência MLP — {label}\n"
            f"Loss treino (azul) | Score validação interna (laranja)"
        )
        ax1.grid(alpha=0.3)
        plt.tight_layout()

        if save:
            slug = label.lower().replace(" ", "_")
            path = self._out / f"mlp_loss_gap_{slug}.png"
            plt.savefig(path, dpi=150, bbox_inches="tight")
            print(f"[LearningCurvePlotter] Loss gap salvo → {path.name}")
        plt.show()
        plt.close()

    def plot_overfit_dashboard(
        self,
        cv_rf: dict[str, tuple[float, float]] | None = None,
        cv_mlp: dict[str, tuple[float, float]] | None = None,
        mlp_pipeline=None,
        train_sizes: np.ndarray | None = None,
        label: str = "Triclasse",
        save: bool = True,
    ) -> None:
        """
        Dashboard completo de diagnóstico de overfit em um único arquivo.

        Inclui: curva de aprendizado + CV RF + CV MLP + loss MLP.

        Parameters
        ----------
        cv_rf         : dict de CV do RF (de TriclassTrainer.cross_validate_rf)
        cv_mlp        : dict de CV do MLP (de TriclassTrainer.cross_validate_mlp)
        mlp_pipeline  : Pipeline(StandardScaler → MLP) treinado
        train_sizes   : pontos da curva de aprendizado (default: 10 pontos)
        """
        if train_sizes is None:
            train_sizes = np.linspace(0.10, 1.0, 8)

        # Calcular curva de aprendizado
        _, train_scores, val_scores = learning_curve(
            self._model, self._X, self._y,
            train_sizes=train_sizes,
            cv=StratifiedKFold(n_splits=3, shuffle=True,
                               random_state=self._random_state),
            scoring=self._scoring,
            n_jobs=-1,
        )
        train_sizes_abs = (train_sizes * len(self._X)).astype(int)

        train_mean = train_scores.mean(axis=1)
        val_mean   = val_scores.mean(axis=1)
        gap        = train_mean - val_mean

        # Número de subplots dinâmico
        n_rows = 2
        n_cols = 2
        fig = plt.figure(figsize=(16, 10))
        gs  = gridspec.GridSpec(n_rows, n_cols, figure=fig,
                                hspace=0.45, wspace=0.35)

        # ── Plot 1: curva de aprendizado ───────────────────────────────────────
        ax1 = fig.add_subplot(gs[0, 0])
        ax1.plot(train_sizes_abs, train_mean, "o-", color="steelblue",
                 label=f"Treino ({self._scoring})")
        ax1.plot(train_sizes_abs, val_mean, "s--", color="darkorange",
                 label="Validação CV")
        ax1.set_xlabel("Amostras de Treino")
        ax1.set_ylabel("Score")
        ax1.set_title("Curva de Aprendizado")
        ax1.legend(fontsize=8)
        ax1.grid(alpha=0.3)

        # ── Plot 2: gap de overfit ─────────────────────────────────────────────
        ax2 = fig.add_subplot(gs[0, 1])
        ax2.plot(train_sizes_abs, gap, "^-", color="crimson", label="Gap")
        ax2.fill_between(train_sizes_abs, 0, gap, alpha=0.15, color="crimson")
        ax2.axhline(0.05, linestyle="--", color="gray", linewidth=0.8)
        ax2.set_xlabel("Amostras de Treino")
        ax2.set_ylabel("Gap (Treino − Val)")
        ax2.set_title("Gap de Overfit\n(> 0.05 = preocupante)")
        ax2.grid(alpha=0.3)

        # ── Plot 3: CV RF ──────────────────────────────────────────────────────
        ax3 = fig.add_subplot(gs[1, 0])
        if cv_rf:
            metrics  = list(cv_rf.keys())
            means_rf = [cv_rf[m][0] for m in metrics]
            stds_rf  = [cv_rf[m][1] for m in metrics]
            x3 = np.arange(len(metrics))
            ax3.bar(x3, means_rf, yerr=stds_rf, capsize=5,
                    color="steelblue", alpha=0.8, ecolor="crimson")
            ax3.set_xticks(x3)
            ax3.set_xticklabels(metrics, rotation=15)
            ax3.set_title("CV Scores — RF\n(±std entre folds)")
            ax3.set_ylim(0, 1.05)
            ax3.grid(axis="y", alpha=0.3)
        else:
            ax3.text(0.5, 0.5, "CV RF não disponível",
                     transform=ax3.transAxes, ha="center")

        # ── Plot 4: CV MLP ─────────────────────────────────────────────────────
        ax4 = fig.add_subplot(gs[1, 1])
        if cv_mlp:
            metrics   = list(cv_mlp.keys())
            means_mlp = [cv_mlp[m][0] for m in metrics]
            stds_mlp  = [cv_mlp[m][1] for m in metrics]
            x4 = np.arange(len(metrics))
            ax4.bar(x4, means_mlp, yerr=stds_mlp, capsize=5,
                    color="darkorange", alpha=0.8, ecolor="crimson")
            ax4.set_xticks(x4)
            ax4.set_xticklabels(metrics, rotation=15)
            ax4.set_title("CV Scores — MLP Pipeline\n(±std entre folds)")
            ax4.set_ylim(0, 1.05)
            ax4.grid(axis="y", alpha=0.3)
        else:
            ax4.text(0.5, 0.5, "CV MLP não disponível",
                     transform=ax4.transAxes, ha="center")

        # Diagnóstico no título geral
        last_gap = gap[-1]
        diag = "OVERFIT detectado" if last_gap > 0.05 else "Sem overfit detectado"
        color = "crimson" if last_gap > 0.05 else "darkgreen"

        fig.suptitle(
            f"Dashboard de Diagnóstico de Overfit — {label}\n{diag}  |  "
            f"Gap final={last_gap:.4f}",
            fontsize=12, fontweight="bold", color=color,
        )

        if save:
            slug = label.lower().replace(" ", "_")
            path = self._out / f"overfit_dashboard_{slug}.png"
            plt.savefig(path, dpi=150, bbox_inches="tight")
            print(f"[LearningCurvePlotter] Dashboard salvo → {path.name}")
        plt.show()
        plt.close()


# ── CLI ────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(
        description="Plota curvas de aprendizado para diagnóstico de overfit."
    )
    parser.add_argument("--model-path", required=True,
                        help="Caminho para o .joblib do modelo treinado")
    parser.add_argument("--scoring", default="f1_macro",
                        help="Métrica de scoring (default: f1_macro)")
    parser.add_argument("--cv", type=int, default=5,
                        help="Número de folds do CV (default: 5)")
    parser.add_argument("--label", default="Modelo",
                        help="Nome do modelo para os gráficos")

    args = parser.parse_args()

    import joblib
    model = joblib.load(args.model_path)
    print(f"Modelo carregado: {args.model_path}")
    print("Para usar o plotter, instancie LearningCurvePlotter(model, X_train, y_train)")
    print("e chame .plot_learning_curve(), .plot_cv_scores() ou .plot_overfit_dashboard()")


### Arquivo: `ml/utils/metrics_logger.py`


In [ ]:
"""
Registro persistente de métricas de treinamento.

Salva os resultados de cada execução do pipeline em um arquivo JSON
(metrics_history.json) e exporta para CSV quando necessário.

Cada entrada inclui: timestamp, identificador do experimento, hiperparâmetros,
métricas de avaliação e metadados do dataset. Isso permite comparar múltiplas
runs, versões do modelo e experimentos ao longo do tempo.

Uso:

    logger = MetricsLogger()
    logger.log(
        result=result_baseline,
        run_id="baseline_v1",
        params={"hidden_layer_sizes": (128, 64), "alpha": 0.0001},
        dataset_info={"n_train": 46172, "n_test": 57110, "n_duplicates_removed": 87084},
    )
    logger.to_csv()
"""

from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path
from typing import Any

import pandas as pd


METRICS_FILE = OUTPUTS_DIR / "metrics_history.json"


class MetricsLogger:
    """
    Registra e persiste métricas de múltiplos experimentos em JSON + CSV.
    """

    def __init__(self, path: Path | str = METRICS_FILE) -> None:
        self._path = Path(path)
        self._path.parent.mkdir(parents=True, exist_ok=True)
        self._history: list[dict] = self._load()

    # ── API pública ────────────────────────────────────────────────────────────

    def log(
        self,
        result: EvaluationResult,
        run_id: str | None = None,
        params: dict | None = None,
        dataset_info: dict | None = None,
        notes: str = "",
    ) -> dict:
        """
        Registra uma avaliação e persiste no arquivo JSON.

        Parameters
        ----------
        result       : EvaluationResult do ModelEvaluator
        run_id       : identificador único da run (ex: "baseline_v1", "tuned_run3")
                       se None, gera automaticamente com timestamp
        params       : hiperparâmetros do modelo (dict)
        dataset_info : informações do dataset (n_train, n_test, etc.)
        notes        : observações livres sobre a run

        Returns
        -------
        dict com o registro salvo.
        """
        ts = datetime.now()
        entry: dict[str, Any] = {
            "run_id":    run_id or f"run_{ts.strftime('%Y%m%d_%H%M%S')}",
            "timestamp": ts.isoformat(),
            "label":     result.label,

            # Métricas de avaliação
            "metrics": {
                "accuracy":   round(result.accuracy,  6),
                "precision":  round(result.precision, 6),
                "recall":     round(result.recall,    6),
                "f1":         round(result.f1,        6),
                "mcc":        round(result.mcc,       6),
                "gm":         round(result.gm,        6),
                "roc_auc":    round(result.roc_auc,   6),
            },

            # Matriz de confusão
            "confusion_matrix": {
                "tp": result.tp,
                "tn": result.tn,
                "fp": result.fp,
                "fn": result.fn,
            },

            # Métricas percentuais (para leitura humana)
            "metrics_pct": {
                "accuracy_pct":  round(result.accuracy  * 100, 4),
                "precision_pct": round(result.precision * 100, 4),
                "recall_pct":    round(result.recall    * 100, 4),
                "f1_pct":        round(result.f1        * 100, 4),
            },

            # Hiperparâmetros e contexto
            "params":       params or {},
            "dataset_info": dataset_info or {},
            "notes":        notes,
        }

        self._history.append(entry)
        self._save()

        print(f"[MetricsLogger] Run '{entry['run_id']}' registrada → {self._path}")
        return entry

    def to_csv(self, path: Path | str | None = None) -> Path:
        """
        Exporta o histórico completo para CSV.

        Cada linha = uma run. Métricas são colunas achatadas.

        Returns
        -------
        Path do CSV gerado.
        """
        if not self._history:
            print("[MetricsLogger] Nenhuma run registrada ainda.")
            return Path()

        rows = []
        for entry in self._history:
            row: dict[str, Any] = {
                "run_id":    entry["run_id"],
                "timestamp": entry["timestamp"],
                "label":     entry["label"],
                "notes":     entry.get("notes", ""),
            }
            row.update(entry.get("metrics", {}))
            row.update({f"cm_{k}": v for k, v in entry.get("confusion_matrix", {}).items()})
            row.update({f"param_{k}": v for k, v in entry.get("params", {}).items()})
            row.update({f"data_{k}": v for k, v in entry.get("dataset_info", {}).items()})
            rows.append(row)

        df = pd.DataFrame(rows)
        csv_path = Path(path) if path else self._path.with_suffix(".csv")
        df.to_csv(csv_path, index=False)
        print(f"[MetricsLogger] CSV exportado → {csv_path}  ({len(df)} runs)")
        return csv_path

    def summary(self) -> pd.DataFrame:
        """
        Retorna um DataFrame resumido com as métricas principais de cada run.
        """
        if not self._history:
            return pd.DataFrame()

        rows = []
        for e in self._history:
            m = e.get("metrics", {})
            cm = e.get("confusion_matrix", {})
            rows.append({
                "run_id":    e["run_id"],
                "label":     e["label"],
                "timestamp": e["timestamp"][:19],
                "accuracy":  f"{m.get('accuracy',0)*100:.4f}%",
                "precision": f"{m.get('precision',0)*100:.4f}%",
                "recall":    f"{m.get('recall',0)*100:.4f}%",
                "f1":        f"{m.get('f1',0)*100:.4f}%",
                "mcc":       f"{m.get('mcc',0):.4f}",
                "roc_auc":   f"{m.get('roc_auc',0):.4f}",
                "tp":  cm.get("tp", ""),
                "tn":  cm.get("tn", ""),
                "fp":  cm.get("fp", ""),
                "fn":  cm.get("fn", ""),
            })

        df = pd.DataFrame(rows)
        return df

    def print_summary(self) -> None:
        """Exibe tabela resumida no terminal."""
        df = self.summary()
        if df.empty:
            print("[MetricsLogger] Nenhuma run registrada.")
            return
        print("\n" + "=" * 100)
        print(f"  Histórico de Experimentos — {len(df)} run(s)")
        print("=" * 100)
        print(df.to_string(index=False))
        print("=" * 100)

    def __len__(self) -> int:
        return len(self._history)

    # ── Métodos privados ───────────────────────────────────────────────────────

    def _load(self) -> list[dict]:
        if self._path.exists():
            try:
                with open(self._path) as f:
                    data = json.load(f)
                print(f"[MetricsLogger] {len(data)} run(s) carregada(s) de {self._path}")
                return data
            except (json.JSONDecodeError, KeyError):
                print(f"[MetricsLogger] Arquivo corrompido — iniciando histórico novo.")
        return []

    def _save(self) -> None:
        with open(self._path, "w") as f:
            json.dump(self._history, f, indent=2, ensure_ascii=False)


### Arquivo: `ml/utils/metrics_plotter.py`


In [ ]:
"""
Visualização comparativa de múltiplas runs de treinamento.

Lê o histórico salvo pelo MetricsLogger e gera gráficos para:
  1. Evolução das métricas ao longo das runs
  2. Comparação lado a lado de dois modelos
  3. Radar chart com todas as métricas de uma run
  4. Dashboard completo (todos os gráficos em um único arquivo)

Uso direto (CLI):
    python -m ml.utils.metrics_plotter               # dashboard completo
    python -m ml.utils.metrics_plotter --compare baseline_v1 tuned_v1

Uso como módulo:
    plotter = MetricsPlotter()
    plotter.plot_evolution()
    plotter.plot_comparison("baseline_v1", "tuned_v1")
    plotter.plot_radar("baseline_v1")
    plotter.plot_dashboard()
"""

from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd


METRICS_FILE = OUTPUTS_DIR / "metrics_history.json"

# Métricas exibidas nos gráficos (em %)
DISPLAY_METRICS = ["accuracy", "precision", "recall", "f1", "mcc", "gm", "roc_auc"]
DISPLAY_LABELS  = ["Acurácia", "Precisão", "Recall", "F1", "MCC", "G-Mean", "ROC-AUC"]
COLORS = plt.rcParams["axes.prop_cycle"].by_key()["color"]


class MetricsPlotter:
    """
    Gera gráficos comparativos a partir do histórico de métricas.
    """

    def __init__(self, metrics_file: Path | str = METRICS_FILE) -> None:
        self._path    = Path(metrics_file)
        self._history = self._load()
        OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

    # ── API pública ────────────────────────────────────────────────────────────

    def plot_evolution(self, save: bool = True) -> None:
        """
        Gráfico de linhas: evolução das métricas ao longo das runs.
        Útil para acompanhar melhoras iterativas entre experimentos.
        """
        if len(self._history) < 2:
            print("[MetricsPlotter] plot_evolution requer ≥ 2 runs. "
                  "Execute o pipeline mais vezes com diferentes parâmetros.")
            return

        df = self._to_dataframe()
        run_labels = df["run_id"].tolist()
        x = np.arange(len(run_labels))

        fig, axes = plt.subplots(2, 1, figsize=(12, 8))

        # Plot principal: accuracy, f1, recall, precision
        ax = axes[0]
        for metric, label, color in zip(
            ["accuracy", "f1", "recall", "precision"],
            ["Acurácia", "F1-Score", "Recall", "Precisão"],
            COLORS[:4],
        ):
            vals = df[metric].values * 100
            ax.plot(x, vals, marker="o", label=label, color=color)
            for xi, v in zip(x, vals):
                ax.annotate(f"{v:.2f}%", (xi, v), textcoords="offset points",
                            xytext=(0, 6), ha="center", fontsize=7)

        ax.set_xticks(x)
        ax.set_xticklabels(run_labels, rotation=30, ha="right")
        ax.set_ylabel("Score (%)")
        ax.set_title("Evolução das Métricas Principais por Run")
        ax.legend(loc="lower right")
        ax.set_ylim(max(0, df["accuracy"].min() * 100 - 2), 101)
        ax.grid(alpha=0.3)

        # Plot secundário: MCC, G-Mean, ROC-AUC
        ax2 = axes[1]
        for metric, label, color in zip(
            ["mcc", "gm", "roc_auc"],
            ["MCC", "G-Mean", "ROC-AUC"],
            COLORS[4:7],
        ):
            vals = df[metric].values
            ax2.plot(x, vals, marker="s", label=label, color=color)
            for xi, v in zip(x, vals):
                ax2.annotate(f"{v:.4f}", (xi, v), textcoords="offset points",
                             xytext=(0, 6), ha="center", fontsize=7)

        ax2.set_xticks(x)
        ax2.set_xticklabels(run_labels, rotation=30, ha="right")
        ax2.set_ylabel("Score (0–1)")
        ax2.set_title("Evolução de MCC, G-Mean e ROC-AUC por Run")
        ax2.legend(loc="lower right")
        ax2.set_ylim(max(0, df["mcc"].min() - 0.05), 1.02)
        ax2.grid(alpha=0.3)

        plt.tight_layout()
        if save:
            path = OUTPUTS_DIR / "metrics_evolution.png"
            plt.savefig(path, dpi=150, bbox_inches="tight")
            print(f"[MetricsPlotter] Evolução salva → {path}")
        plt.show()
        plt.close()

    def plot_comparison(
        self,
        run_id_a: str,
        run_id_b: str,
        save: bool = True,
    ) -> None:
        """
        Barras agrupadas comparando duas runs lado a lado.

        Ideal para comparar baseline vs. otimizado.
        """
        a = self._get_entry(run_id_a)
        b = self._get_entry(run_id_b)
        if a is None or b is None:
            return

        metrics_a = [a["metrics"][m] * 100 if m != "mcc" else a["metrics"][m]
                     for m in DISPLAY_METRICS]
        metrics_b = [b["metrics"][m] * 100 if m != "mcc" else b["metrics"][m]
                     for m in DISPLAY_METRICS]

        x = np.arange(len(DISPLAY_METRICS))
        w = 0.35

        fig, ax = plt.subplots(figsize=(13, 6))
        bars_a = ax.bar(x - w/2, metrics_a, w, label=a.get("label", run_id_a),
                        color=COLORS[0], alpha=0.85)
        bars_b = ax.bar(x + w/2, metrics_b, w, label=b.get("label", run_id_b),
                        color=COLORS[1], alpha=0.85)

        for bar, val in zip(bars_a, metrics_a):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f"{val:.2f}", ha="center", va="bottom", fontsize=8)
        for bar, val in zip(bars_b, metrics_b):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f"{val:.2f}", ha="center", va="bottom", fontsize=8)

        ax.set_xticks(x)
        ax.set_xticklabels(DISPLAY_LABELS)
        ax.set_ylabel("Score (% ou 0–1 para MCC)")
        ax.set_title(f"Comparação: {a.get('label', run_id_a)} vs. {b.get('label', run_id_b)}")
        ax.legend()
        ax.set_ylim(min(min(metrics_a), min(metrics_b)) - 2, 102)
        ax.grid(axis="y", alpha=0.3)

        plt.tight_layout()
        if save:
            slug = f"{run_id_a}_vs_{run_id_b}".replace(" ", "_")
            path = OUTPUTS_DIR / f"comparison_{slug}.png"
            plt.savefig(path, dpi=150, bbox_inches="tight")
            print(f"[MetricsPlotter] Comparação salva → {path}")
        plt.show()
        plt.close()

    def plot_radar(self, run_id: str, save: bool = True) -> None:
        """
        Radar chart (spider plot) com todas as métricas de uma run.

        Visualização intuitiva da "forma" do desempenho — fácil de ver
        onde o modelo é forte e onde há espaço para melhora.
        """
        entry = self._get_entry(run_id)
        if entry is None:
            return

        values = [entry["metrics"][m] for m in DISPLAY_METRICS]
        # Fechar o polígono
        values_closed   = values + [values[0]]
        labels_for_plot = DISPLAY_LABELS + [DISPLAY_LABELS[0]]
        N = len(DISPLAY_METRICS)
        angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
        angles_closed = angles + [angles[0]]

        fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"polar": True})

        ax.plot(angles_closed, values_closed, "o-", linewidth=2, color=COLORS[0])
        ax.fill(angles_closed, values_closed, alpha=0.25, color=COLORS[0])

        ax.set_xticks(angles)
        ax.set_xticklabels(DISPLAY_LABELS, fontsize=11)
        ax.set_ylim(0, 1.05)
        ax.set_yticks([0.5, 0.7, 0.9, 1.0])
        ax.set_yticklabels(["50%", "70%", "90%", "100%"], fontsize=8)
        ax.set_title(f"Radar de Métricas — {entry.get('label', run_id)}\n"
                     f"({entry['timestamp'][:10]})", pad=20)

        # Anotar valores
        for angle, value, label in zip(angles, values, DISPLAY_LABELS):
            ax.annotate(
                f"{value:.4f}",
                xy=(angle, value),
                xytext=(angle, value + 0.05),
                ha="center", va="center", fontsize=8,
            )

        if save:
            slug = run_id.replace(" ", "_")
            path = OUTPUTS_DIR / f"radar_{slug}.png"
            plt.savefig(path, dpi=150, bbox_inches="tight")
            print(f"[MetricsPlotter] Radar salvo → {path}")
        plt.show()
        plt.close()

    def plot_confusion_heatmap(self, run_id: str, save: bool = True) -> None:
        """
        Heatmap da matriz de confusão a partir do JSON histórico.
        """
        entry = self._get_entry(run_id)
        if entry is None:
            return

        cm_dict = entry.get("confusion_matrix", {})
        tn = cm_dict.get("tn", 0)
        fp = cm_dict.get("fp", 0)
        fn = cm_dict.get("fn", 0)
        tp = cm_dict.get("tp", 0)

        cm = np.array([[tn, fp], [fn, tp]])
        total = cm.sum()

        fig, ax = plt.subplots(figsize=(7, 5))
        im = ax.imshow(cm, cmap="Blues")
        plt.colorbar(im, ax=ax)

        labels = ["Benigno", "Ataque DDoS"]
        ax.set_xticks([0, 1]); ax.set_xticklabels(labels)
        ax.set_yticks([0, 1]); ax.set_yticklabels(labels)
        ax.set_xlabel("Predição")
        ax.set_ylabel("Real")
        ax.set_title(f"Matriz de Confusão — {entry.get('label', run_id)}")

        for i in range(2):
            for j in range(2):
                val   = cm[i, j]
                pct   = val / total * 100
                color = "white" if cm[i, j] > cm.max() / 2 else "black"
                ax.text(j, i, f"{val:,}\n({pct:.2f}%)",
                        ha="center", va="center", color=color, fontsize=11)

        plt.tight_layout()
        if save:
            slug = run_id.replace(" ", "_")
            path = OUTPUTS_DIR / f"cm_history_{slug}.png"
            plt.savefig(path, dpi=150, bbox_inches="tight")
            print(f"[MetricsPlotter] Heatmap salvo → {path}")
        plt.show()
        plt.close()

    def plot_dashboard(self, save: bool = True) -> None:
        """
        Dashboard completo: evolução + tabela de resumo em um único arquivo PNG.
        """
        df = self._to_dataframe()
        if df.empty:
            print("[MetricsPlotter] Nenhuma run registrada.")
            return

        n_runs = len(df)
        fig = plt.figure(figsize=(16, 10))
        gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.3)

        # ── Plot 1: barras agrupadas de F1/Recall/Precision por run ──────────
        ax1 = fig.add_subplot(gs[0, :])
        x = np.arange(n_runs)
        w = 0.25
        metrics_to_bar = [
            ("f1",        "F1-Score",  COLORS[0]),
            ("recall",    "Recall",    COLORS[1]),
            ("precision", "Precisão",  COLORS[2]),
        ]
        for i, (metric, label, color) in enumerate(metrics_to_bar):
            vals = df[metric].values * 100
            bars = ax1.bar(x + (i - 1) * w, vals, w, label=label, color=color, alpha=0.85)
            for bar, v in zip(bars, vals):
                ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                         f"{v:.2f}%", ha="center", va="bottom", fontsize=7)

        ax1.set_xticks(x)
        ax1.set_xticklabels(df["run_id"].tolist(), rotation=20, ha="right")
        ax1.set_ylabel("Score (%)")
        ax1.set_title("F1 / Recall / Precisão por Experimento")
        ax1.legend()
        ax1.set_ylim(max(0, df["f1"].min() * 100 - 3), 101)
        ax1.grid(axis="y", alpha=0.3)

        # ── Plot 2: MCC ao longo das runs ─────────────────────────────────────
        ax2 = fig.add_subplot(gs[1, 0])
        ax2.plot(x, df["mcc"].values, "o-", color=COLORS[3], linewidth=2)
        for xi, v in zip(x, df["mcc"].values):
            ax2.annotate(f"{v:.4f}", (xi, v), xytext=(0, 8),
                         textcoords="offset points", ha="center", fontsize=8)
        ax2.set_xticks(x)
        ax2.set_xticklabels(df["run_id"].tolist(), rotation=25, ha="right")
        ax2.set_ylabel("MCC (0–1)")
        ax2.set_title("MCC por Experimento")
        ax2.set_ylim(max(0, df["mcc"].min() - 0.05), 1.02)
        ax2.grid(alpha=0.3)

        # ── Plot 3: FP e FN (erros) ───────────────────────────────────────────
        ax3 = fig.add_subplot(gs[1, 1])
        w2 = 0.35
        fp_vals = df["cm_fp"].values if "cm_fp" in df.columns else np.zeros(n_runs)
        fn_vals = df["cm_fn"].values if "cm_fn" in df.columns else np.zeros(n_runs)
        ax3.bar(x - w2/2, fp_vals, w2, label="Falsos Positivos (alarmes)", color=COLORS[4], alpha=0.85)
        ax3.bar(x + w2/2, fn_vals, w2, label="Falsos Negativos (ataques perdidos)", color="crimson", alpha=0.85)
        for xi, v in zip(x, fp_vals):
            ax3.text(xi - w2/2, v + 0.3, str(int(v)), ha="center", fontsize=9)
        for xi, v in zip(x, fn_vals):
            ax3.text(xi + w2/2, v + 0.3, str(int(v)), ha="center", fontsize=9)
        ax3.set_xticks(x)
        ax3.set_xticklabels(df["run_id"].tolist(), rotation=25, ha="right")
        ax3.set_ylabel("Contagem")
        ax3.set_title("Erros de Classificação (FP vs FN)")
        ax3.legend(fontsize=8)
        ax3.grid(axis="y", alpha=0.3)

        fig.suptitle("Dashboard de Métricas — Experimentos DDoS MLP (InSDN8)",
                     fontsize=14, fontweight="bold")

        if save:
            path = OUTPUTS_DIR / "dashboard_metrics.png"
            plt.savefig(path, dpi=150, bbox_inches="tight")
            print(f"[MetricsPlotter] Dashboard salvo → {path}")
        plt.show()
        plt.close()

    def list_runs(self) -> None:
        """Lista todas as runs registradas."""
        if not self._history:
            print("[MetricsPlotter] Nenhuma run registrada.")
            return
        print(f"\n{'Run ID':<30} {'Label':<25} {'Timestamp':<20} {'F1':>8} {'Recall':>8}")
        print("-" * 95)
        for e in self._history:
            m = e.get("metrics", {})
            print(f"{e['run_id']:<30} {e.get('label',''):<25} "
                  f"{e['timestamp'][:19]:<20} "
                  f"{m.get('f1',0)*100:>7.4f}% "
                  f"{m.get('recall',0)*100:>7.4f}%")

    # ── Métodos privados ───────────────────────────────────────────────────────

    def _load(self) -> list[dict]:
        if self._path.exists():
            with open(self._path) as f:
                return json.load(f)
        return []

    def _to_dataframe(self) -> pd.DataFrame:
        if not self._history:
            return pd.DataFrame()
        rows = []
        for e in self._history:
            row = {"run_id": e["run_id"], "label": e.get("label", ""), "timestamp": e["timestamp"]}
            row.update(e.get("metrics", {}))
            row.update({f"cm_{k}": v for k, v in e.get("confusion_matrix", {}).items()})
            rows.append(row)
        return pd.DataFrame(rows)

    def _get_entry(self, run_id: str) -> dict | None:
        for e in self._history:
            if e["run_id"] == run_id:
                return e
        print(f"[MetricsPlotter] Run '{run_id}' não encontrada.")
        print(f"  Runs disponíveis: {[e['run_id'] for e in self._history]}")
        return None


# ── CLI ────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(description="Plota métricas do histórico de experimentos.")
    parser.add_argument("--list",    action="store_true", help="Lista todas as runs")
    parser.add_argument("--evolve",  action="store_true", help="Plota evolução das métricas")
    parser.add_argument("--dashboard", action="store_true", help="Dashboard completo")
    parser.add_argument("--radar",   metavar="RUN_ID", help="Radar chart de uma run")
    parser.add_argument("--compare", nargs=2, metavar=("RUN_A", "RUN_B"),
                        help="Comparação lado a lado entre duas runs")

    args = parser.parse_args()
    plotter = MetricsPlotter()

    if args.list:
        plotter.list_runs()
    elif args.evolve:
        plotter.plot_evolution()
    elif args.radar:
        plotter.plot_radar(args.radar)
    elif args.compare:
        plotter.plot_comparison(args.compare[0], args.compare[1])
    else:
        # padrão: dashboard completo
        plotter.plot_dashboard()


### Arquivo: `ml/data/loader.py`


In [ ]:
"""
Camada de carregamento de dados.

Define o protocolo DataLoader (ISP/DIP — depender de abstrações) e a
implementação concreta InSDNLoader para o dataset insdn8_ddos_binary_0n1d.csv.

O dataset insdn8 é um subconjunto pré-selecionado do InSDN original
(Elsayed et al., IEEE Access 2020) com 8 features de fluxo de rede e
classificação binária: 0 = Benigno, 1 = Ataque DDoS.

Avaliação do dataset:
  - Arquivo : insdn8_ddos_binary_0n1d.csv
  - Instâncias: ~190.366 (compatível com o plano — InSDN original: 361.317)
  - Features   : 8 (já pré-selecionadas — Protocol, Flow Duration, Flow IAT Max,
                    Bwd Pkts/s, Pkt Len Std, Pkt Len Var, Bwd IAT Tot, Flow Pkts/s)
  - Label      : binária (0 = Normal, 1 = DDoS) — já binarizada
  - Desbalanceamento: majoritariamente classe 1 (ataque) — SMOTE necessário
  - Nota: a seleção de features via SHAP é realizada mesmo com poucas features
    para confirmar importância relativa e descartar features de variância zero.
"""

from __future__ import annotations

from pathlib import Path
from typing import Protocol

import pandas as pd



# ── Protocolo (ISP + DIP) ──────────────────────────────────────────────────────

class DataLoader(Protocol):
    """
    Interface mínima para carregadores de dataset.

    Qualquer implementação que retorne (X, y) como DataFrames é válida.
    Seguindo DIP, os módulos de pipeline dependem desta abstração,
    não de InSDNLoader diretamente.
    """

    def load(self) -> tuple[pd.DataFrame, pd.Series]:
        """
        Carrega o dataset e retorna (features, alvo).

        Returns
        -------
        X : pd.DataFrame
            Features brutas (sem transformações).
        y : pd.Series
            Coluna alvo com valores inteiros 0/1.
        """
        ...

    def describe(self) -> None:
        """Exibe resumo do dataset no stdout para fins de EDA inicial."""
        ...


# ── Implementação concreta ─────────────────────────────────────────────────────

class InSDNLoader:
    """
    Carrega e valida o dataset InSDN8 (insdn8_ddos_binary_0n1d.csv).

    Responsabilidade única: ler o CSV e retornar (X, y) sem nenhuma
    transformação — o split e o pré-processamento ficam em outras classes.
    """

    def __init__(self, path: Path | str = DATASET_PATH) -> None:
        self._path = Path(path)

    # ── API pública ────────────────────────────────────────────────────────────

    def load(self) -> tuple[pd.DataFrame, pd.Series]:
        """
        Carrega o CSV e retorna (X, y).

        EDA mínima embutida: verifica se o arquivo existe, se a coluna alvo
        está presente e se os valores são binários (0/1).

        Returns
        -------
        X : pd.DataFrame  — features brutas
        y : pd.Series     — alvo binário (0 = Benigno, 1 = Ataque)

        Raises
        ------
        FileNotFoundError  se o CSV não existir no caminho configurado.
        ValueError         se a coluna alvo estiver ausente ou não for binária.
        """
        if not self._path.exists():
            raise FileNotFoundError(
                f"Dataset não encontrado: {self._path}\n"
                f"Verifique ml/config.py → DATASET_PATH."
            )

        print(f"[InSDNLoader] Carregando: {self._path}")
        data = pd.read_csv(self._path, low_memory=False)
        print(f"  Shape bruto: {data.shape}")

        self._validate(data)

        X = data.drop(columns=[TARGET_COL])
        y = data[TARGET_COL].astype(int)

        return X, y

    def describe(self) -> None:
        """EDA textual: shape, tipos, distribuição do alvo, ausentes, infinitos."""
        import numpy as np

        X, y = self.load()
        data = X.copy()
        data[TARGET_COL] = y

        print("\n" + "=" * 60)
        print("  EDA — InSDN8 Dataset")
        print("=" * 60)

        print(f"\nShape       : {data.shape}")
        print(f"Memória     : {data.memory_usage(deep=True).sum() / 1e6:.2f} MB")

        print("\n── Tipos de dados ──")
        print(data.dtypes.to_string())

        print("\n── Distribuição do alvo ──")
        counts = y.value_counts()
        pcts   = y.value_counts(normalize=True).mul(100).round(2)
        for cls in counts.index:
            label = "Benigno" if cls == 0 else "Ataque DDoS"
            print(f"  {cls} ({label}): {counts[cls]:>7,}  ({pcts[cls]:.2f}%)")

        bal = self._shannon_balance(y)
        print(f"\n  Entropia de Shannon (balanceamento): {bal:.4f}")
        print(f"  → 0 = totalmente desbalanceado | 1 = perfeitamente balanceado")

        print("\n── Valores ausentes (antes do split) ──")
        missing = data.isnull().sum()
        if missing.any():
            print(missing[missing > 0].to_string())
        else:
            print("  Nenhum valor ausente.")

        print("\n── Valores infinitos (antes do split) ──")
        inf_count = np.isinf(data.select_dtypes(include=np.number)).sum().sum()
        print(f"  Total de Inf: {inf_count}")

        print("\n── Duplicatas (antes do split) ──")
        print(f"  Linhas duplicadas: {data.duplicated().sum():,}")

        print("\n── Estatísticas descritivas (features) ──")
        print(data.drop(columns=[TARGET_COL]).describe().to_string())
        print("=" * 60 + "\n")

    # ── Métodos privados ───────────────────────────────────────────────────────

    def _validate(self, data: pd.DataFrame) -> None:
        if TARGET_COL not in data.columns:
            raise ValueError(
                f"Coluna alvo '{TARGET_COL}' não encontrada no dataset. "
                f"Colunas disponíveis: {list(data.columns)}"
            )
        unique_vals = set(data[TARGET_COL].dropna().unique())
        if not unique_vals.issubset({0, 1, 0.0, 1.0}):
            raise ValueError(
                f"A coluna '{TARGET_COL}' deve conter apenas valores binários (0/1). "
                f"Encontrado: {unique_vals}"
            )

    @staticmethod
    def _shannon_balance(y: pd.Series) -> float:
        """Entropia de Shannon normalizada: 0 (desbalanceado) → 1 (balanceado)."""
        import numpy as np
        n_classes = y.nunique()
        if n_classes < 2:
            return 0.0
        n = len(y)
        H = sum(
            -(count / n) * np.log(count / n)
            for count in y.value_counts()
        )
        return H / np.log(n_classes)


### Arquivo: `ml/preprocessing/balancer.py`


In [ ]:
"""
Balanceamento de classes com SMOTE.

SRP: responsável exclusivamente pelo balanceamento do conjunto de treino.

Regra obrigatória (boas práticas do curso):
  SMOTE gera instâncias SINTÉTICAS — aplique SOMENTE no treino.
  O teste deve representar a distribuição real do mundo, não dados artificiais.

O dataset insdn8 é heavily imbalanced (majoritariamente classe 1 = DDoS),
o que justifica o uso de SMOTE para equilibrar as classes antes do treino.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE



class ClassBalancer:
    """
    Aplica SMOTE para sobreamostrar a classe minoritária no conjunto de treino.

    Uso:
        balancer = ClassBalancer()
        X_bal, y_bal = balancer.fit_resample(X_train_scaled, y_train)
        # X_test NÃO passa pelo balancer
    """

    def __init__(self, random_state: int = RANDOM_STATE) -> None:
        self._smote = SMOTE(random_state=random_state)

    # ── API pública ────────────────────────────────────────────────────────────

    def fit_resample(
        self,
        X: np.ndarray | pd.DataFrame,
        y: np.ndarray | pd.Series,
    ) -> tuple[np.ndarray, np.ndarray]:
        """
        Aplica SMOTE ao treino e retorna arrays balanceados.

        Registra no stdout a distribuição antes/depois para auditoria.

        Returns
        -------
        X_resampled : np.ndarray
        y_resampled : np.ndarray
        """
        y_arr = np.array(y)

        counts_before = {cls: int((y_arr == cls).sum()) for cls in np.unique(y_arr)}
        pcts_before   = {cls: cnt / len(y_arr) * 100 for cls, cnt in counts_before.items()}

        print("[ClassBalancer] Distribuição ANTES do SMOTE:")
        for cls, cnt in counts_before.items():
            label = "Benigno" if cls == 0 else "Ataque DDoS"
            print(f"  {cls} ({label}): {cnt:>7,}  ({pcts_before[cls]:.1f}%)")

        X_res, y_res = self._smote.fit_resample(X, y_arr)

        counts_after = {cls: int((y_res == cls).sum()) for cls in np.unique(y_res)}
        pcts_after   = {cls: cnt / len(y_res) * 100 for cls, cnt in counts_after.items()}

        print("[ClassBalancer] Distribuição DEPOIS do SMOTE:")
        for cls, cnt in counts_after.items():
            label = "Benigno" if cls == 0 else "Ataque DDoS"
            print(f"  {cls} ({label}): {cnt:>7,}  ({pcts_after[cls]:.1f}%)")

        print(f"[ClassBalancer] Shape treino balanceado: {X_res.shape}")

        return X_res, y_res


### Arquivo: `ml/preprocessing/cleaner.py`


In [ ]:
"""
Limpeza e preparação dos dados de treino.

SRP: este módulo lida exclusivamente com a remoção de ruído dos dados
(duplicatas, infinitos, valores ausentes).

Regra de ouro: todas as operações de fit são realizadas APENAS no
conjunto de treino. O teste recebe apenas .transform().
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer



class DataCleaner:
    """
    Remove duplicatas, trata infinitos e imputa valores ausentes.

    Uso correto (evita data leakage):
        cleaner = DataCleaner()
        X_train, y_train = cleaner.fit_transform(X_train, y_train)
        X_test            = cleaner.transform(X_test)
    """

    def __init__(self, strategy: str = IMPUTER_STRATEGY) -> None:
        self._imputer: SimpleImputer = SimpleImputer(strategy=strategy)
        self._columns: list[str] = []
        self._fitted: bool = False

    # ── API pública ────────────────────────────────────────────────────────────

    def fit_transform(
        self,
        X: pd.DataFrame,
        y: pd.Series,
    ) -> tuple[pd.DataFrame, pd.Series]:
        """
        Aplica limpeza completa ao conjunto de TREINO.

        Etapas (nesta ordem):
          1. Remove duplicatas (inclui y para consistência)
          2. Substitui Inf/-Inf por NaN
          3. Fit + transform do imputador de medianas

        Returns
        -------
        X_clean : pd.DataFrame
        y_clean : pd.Series  (alinhado com X após remoção de duplicatas)
        """
        self._columns = X.columns.tolist()

        # 1. Duplicatas — reconstruir temporariamente com target
        df_tmp = X.copy()
        df_tmp["__target__"] = y.values

        before = len(df_tmp)
        df_tmp = df_tmp.drop_duplicates(keep="first")
        after  = len(df_tmp)
        print(f"[DataCleaner] Duplicatas removidas: {before - after:,} "
              f"({before:,} → {after:,})")

        X_clean = df_tmp.drop(columns=["__target__"])
        y_clean = pd.Series(df_tmp["__target__"].values, name=y.name)

        # 2. Infinitos → NaN
        X_clean = X_clean.replace([np.inf, -np.inf], np.nan)
        inf_total = (before - after)  # já contabilizado
        nan_count = X_clean.isnull().sum().sum()
        print(f"[DataCleaner] NaN após substituição de Inf: {nan_count:,}")

        # 3. Imputação (fit SOMENTE no treino)
        self._imputer.fit(X_clean)
        X_imputed = pd.DataFrame(
            self._imputer.transform(X_clean),
            columns=self._columns,
        )
        self._fitted = True

        print(f"[DataCleaner] NaN após imputação (treino): "
              f"{X_imputed.isnull().sum().sum()}")
        print(f"[DataCleaner] Shape final do treino: {X_imputed.shape}")

        return X_imputed, y_clean

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """
        Aplica apenas a transformação ao conjunto de TESTE.

        Não faz fit — usa os parâmetros aprendidos em fit_transform().
        """
        if not self._fitted:
            raise RuntimeError(
                "DataCleaner: chame fit_transform() no treino antes de transform()."
            )

        X_clean = X.replace([np.inf, -np.inf], np.nan)

        X_imputed = pd.DataFrame(
            self._imputer.transform(X_clean),
            columns=self._columns,
        )
        print(f"[DataCleaner] NaN após imputação (teste): "
              f"{X_imputed.isnull().sum().sum()}")
        return X_imputed

    @property
    def imputer(self) -> SimpleImputer:
        """Acesso ao imputador fitado para persistência."""
        return self._imputer


### Arquivo: `ml/preprocessing/scaler.py`


In [ ]:
"""
Escalonamento de features.

SRP: responsável exclusivamente pelo escalonamento via StandardScaler.

Regra absoluta (boas práticas do curso):
  fit() APENAS no treino — transform() em treino e teste.
  MLP é especialmente sensível à escala: sem escalonamento o treino pode divergir.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler


class FeatureScaler:
    """
    Wrapper do StandardScaler que força o uso correto (fit no treino apenas).

    Uso:
        scaler = FeatureScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled  = scaler.transform(X_test)     # sem re-fit
    """

    def __init__(self) -> None:
        self._scaler: StandardScaler = StandardScaler()
        self._columns: list[str] = []
        self._fitted: bool = False

    # ── API pública ────────────────────────────────────────────────────────────

    def fit_transform(self, X: pd.DataFrame | np.ndarray) -> np.ndarray:
        """
        Fit no treino e transforma.

        Returns
        -------
        np.ndarray com as features escalonadas (z-score: média≈0, std≈1).
        """
        if isinstance(X, pd.DataFrame):
            self._columns = X.columns.tolist()

        X_scaled = self._scaler.fit_transform(X)
        self._fitted = True

        mean_of_means = X_scaled.mean(axis=0).mean()
        mean_of_stds  = X_scaled.std(axis=0).mean()
        print(f"[FeatureScaler] Escalonamento aplicado ao treino.")
        print(f"  Média das médias (deve ser ≈0): {mean_of_means:.4f}")
        print(f"  Média dos desvios (deve ser ≈1): {mean_of_stds:.4f}")

        return X_scaled

    def transform(self, X: pd.DataFrame | np.ndarray) -> np.ndarray:
        """
        Aplica escalonamento ao teste com os parâmetros do treino.

        Não faz re-fit — evita data leakage.
        """
        if not self._fitted:
            raise RuntimeError(
                "FeatureScaler: chame fit_transform() no treino antes de transform()."
            )
        return self._scaler.transform(X)

    @property
    def scaler(self) -> StandardScaler:
        """Acesso ao scaler fitado para persistência."""
        return self._scaler


### Arquivo: `ml/features/selector.py`


In [ ]:
"""
Seleção de features em duas etapas:

  1. VarianceThreshold — remove features com variância ≤ threshold (zero info).
  2. SHAP (TreeExplainer) — rankeia features por importância e seleciona top-N.

SRP: este módulo lida apenas com seleção de features.

Nota sobre o dataset insdn8:
  O dataset já contém apenas 8 features pré-selecionadas do InSDN original.
  O VarianceThreshold ainda é executado para detectar constantes acidentais.
  A análise SHAP é mantida para confirmar a importância relativa das features
  e gerar o relatório/plot — mesmo que nenhuma feature seja descartada ao final.
  Se N_FEATURES_TO_SELECT=None (padrão), mantém todas as features sobreviventes.
"""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold



class FeatureSelector:
    """
    Seleciona as features mais importantes usando VarianceThreshold + SHAP.

    Uso correto (evita data leakage — SHAP só vê dados de treino):
        selector = FeatureSelector()
        X_train_sel = selector.fit_transform(X_train, y_train)
        X_test_sel  = selector.transform(X_test)
    """

    def __init__(
        self,
        n_features: int | None = N_FEATURES_TO_SELECT,
        variance_threshold: float = VARIANCE_THRESHOLD,
        shap_sample_size: int = SHAP_SAMPLE_SIZE,
        random_state: int = RANDOM_STATE,
        save_plots: bool = True,
    ) -> None:
        self._n_features        = n_features
        self._var_thresh        = VarianceThreshold(threshold=variance_threshold)
        self._shap_sample_size  = shap_sample_size
        self._random_state      = random_state
        self._save_plots        = save_plots
        self._selected_features: list[str] = []
        self._shap_importance: pd.DataFrame | None = None
        self._fitted: bool = False

    # ── API pública ────────────────────────────────────────────────────────────

    def fit_transform(
        self,
        X: pd.DataFrame,
        y: pd.Series | np.ndarray,
    ) -> pd.DataFrame:
        """
        Ajusta o seletor ao treino e retorna as features selecionadas.

        Etapas:
          1. VarianceThreshold.fit_transform(X_train)
          2. RandomForest + SHAP nos dados de treino (amostra)
          3. Filtra top-N features por importância SHAP
        """
        # ── Etapa 1: VarianceThreshold ─────────────────────────────────────────
        cols_before = X.columns.tolist()
        self._var_thresh.fit(X)

        X_var = pd.DataFrame(
            self._var_thresh.transform(X),
            columns=X.columns[self._var_thresh.get_support()].tolist(),
            index=X.index,
        )

        removed_var = set(cols_before) - set(X_var.columns)
        print(f"[FeatureSelector] VarianceThreshold removeu {len(removed_var)} feature(s):")
        if removed_var:
            for f in sorted(removed_var):
                print(f"  - {f}")
        else:
            print("  (nenhuma feature removida — todas possuem variância > 0)")
        print(f"[FeatureSelector] Features restantes após VarianceThreshold: {X_var.shape[1]}")

        # ── Etapa 2: SHAP com RandomForest ────────────────────────────────────
        importance_df = self._compute_shap_importance(X_var, y)
        self._shap_importance = importance_df

        print("\n[FeatureSelector] Importância SHAP (todas as features):")
        print(importance_df.to_string(index=False))

        # ── Etapa 3: Seleção top-N ─────────────────────────────────────────────
        if self._n_features is not None:
            n = min(self._n_features, len(importance_df))
            self._selected_features = importance_df.head(n)["feature"].tolist()
        else:
            # Manter todas as features que sobreviveram ao VarianceThreshold
            self._selected_features = importance_df["feature"].tolist()

        print(f"\n[FeatureSelector] Features selecionadas ({len(self._selected_features)}):")
        for i, f in enumerate(self._selected_features, 1):
            print(f"  {i:2d}. {f}")

        if self._save_plots:
            self._save_shap_plot(X_var, y)

        self._fitted = True
        return X_var[self._selected_features].reset_index(drop=True)

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """
        Aplica a seleção de features ao conjunto de TESTE.

        Não recalcula SHAP — usa as features selecionadas no fit_transform().
        """
        if not self._fitted:
            raise RuntimeError(
                "FeatureSelector: chame fit_transform() no treino antes de transform()."
            )

        X_var = pd.DataFrame(
            self._var_thresh.transform(X),
            columns=X.columns[self._var_thresh.get_support()].tolist(),
            index=X.index,
        )
        return X_var[self._selected_features].reset_index(drop=True)

    @property
    def selected_features(self) -> list[str]:
        """Lista de features selecionadas (disponível após fit_transform)."""
        if not self._fitted:
            raise RuntimeError("FeatureSelector ainda não foi ajustado.")
        return self._selected_features.copy()

    @property
    def variance_filter(self) -> VarianceThreshold:
        """Acesso ao VarianceThreshold fitado para persistência."""
        return self._var_thresh

    @property
    def shap_importance(self) -> pd.DataFrame | None:
        """DataFrame com importâncias SHAP calculadas no treino."""
        return self._shap_importance

    # ── Métodos privados ───────────────────────────────────────────────────────

    def _compute_shap_importance(
        self,
        X: pd.DataFrame,
        y: pd.Series | np.ndarray,
    ) -> pd.DataFrame:
        """
        Treina RandomForest rápido e calcula SHAP values.

        Usa amostra aleatória (SHAP_SAMPLE_SIZE) para ser eficiente.
        TreeExplainer é exato para modelos baseados em árvore.
        """
        try:
            import shap
        except ImportError:
            print("[FeatureSelector] shap não instalado — usando feature_importances_ como fallback.")
            return self._fallback_importance(X, y)

        sample_size = min(self._shap_sample_size, len(X))
        idx = np.random.RandomState(self._random_state).choice(len(X), sample_size, replace=False)
        X_sample = X.iloc[idx]
        y_sample = np.array(y)[idx]

        print(f"\n[FeatureSelector] Treinando RandomForest auxiliar para SHAP "
              f"(amostra: {sample_size:,})...")
        rf = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            random_state=self._random_state,
            n_jobs=-1,
        )
        rf.fit(X_sample, y_sample)

        print("[FeatureSelector] Calculando SHAP values...")
        explainer   = shap.TreeExplainer(rf)
        shap_values = explainer.shap_values(X_sample)

        # Compatibilidade com diferentes versões do SHAP:
        #   SHAP < 0.41 : lista de arrays 2D  → shap_values[1] shape (n, f)
        #   SHAP >= 0.41: array 3D             → shap_values shape (n, f, classes)
        if isinstance(shap_values, list):
            # versão antiga: lista [classe_0, classe_1]
            shap_vals = shap_values[1]
        elif shap_values.ndim == 3:
            # versão nova: (n_samples, n_features, n_classes) — pegar classe 1
            shap_vals = shap_values[:, :, 1]
        else:
            # array 2D direto (alguns modelos retornam isso)
            shap_vals = shap_values

        importances = np.abs(shap_vals).mean(axis=0)
        df = pd.DataFrame({
            "feature":          X.columns.tolist(),
            "shap_importance":  importances,
        }).sort_values("shap_importance", ascending=False).reset_index(drop=True)

        return df

    def _fallback_importance(
        self,
        X: pd.DataFrame,
        y: pd.Series | np.ndarray,
    ) -> pd.DataFrame:
        """Fallback: usa feature_importances_ do RandomForest quando SHAP não disponível."""
        rf = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            random_state=self._random_state,
            n_jobs=-1,
        )
        rf.fit(X, y)
        df = pd.DataFrame({
            "feature":          X.columns.tolist(),
            "shap_importance":  rf.feature_importances_,
        }).sort_values("shap_importance", ascending=False).reset_index(drop=True)
        return df

    def _save_shap_plot(self, X: pd.DataFrame, y: pd.Series | np.ndarray) -> None:
        """Gera e salva o beeswarm plot do SHAP em outputs/."""
        try:
            import shap
            import matplotlib.pyplot as plt

            sample_size = min(self._shap_sample_size, len(X))
            idx = np.random.RandomState(self._random_state).choice(len(X), sample_size, replace=False)
            X_sample = X.iloc[idx]
            y_sample = np.array(y)[idx]

            rf = RandomForestClassifier(
                n_estimators=100, max_depth=10,
                random_state=self._random_state, n_jobs=-1,
            )
            rf.fit(X_sample, y_sample)
            explainer   = shap.TreeExplainer(rf)
            shap_values = explainer.shap_values(X_sample)
            if isinstance(shap_values, list):
                vals = shap_values[1]
            elif shap_values.ndim == 3:
                vals = shap_values[:, :, 1]
            else:
                vals = shap_values

            OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

            # Beeswarm plot
            shap.summary_plot(vals, X_sample, max_display=20, show=False)
            plt.title("SHAP Summary — InSDN8 (treino)")
            plt.tight_layout()
            plt.savefig(OUTPUTS_DIR / "shap_beeswarm.png", dpi=150, bbox_inches="tight")
            plt.close()

            # Bar plot
            shap.summary_plot(vals, X_sample, plot_type="bar", max_display=20, show=False)
            plt.title("SHAP Feature Importance — InSDN8 (treino)")
            plt.tight_layout()
            plt.savefig(OUTPUTS_DIR / "shap_bar.png", dpi=150, bbox_inches="tight")
            plt.close()

            print(f"[FeatureSelector] Plots SHAP salvos em {OUTPUTS_DIR}/")

        except Exception as e:
            print(f"[FeatureSelector] Não foi possível salvar plots SHAP: {e}")


### Arquivo: `ml/models/mlp_model.py`


In [ ]:
"""
Modelo MLP para detecção de DDoS.

SRP: este módulo encapsula exclusivamente a definição e configuração do
MLPClassifier. A lógica de treino, avaliação e persistência fica em
módulos separados.

Arquitetura: conforme Mehmood et al. (PLoS ONE, 2025)
  Input → Dense(128, ReLU) → Dense(64, ReLU) → Output(Softmax binário)
  Solver: ADAM, regularização L2, early stopping.
"""

from __future__ import annotations

from sklearn.neural_network import MLPClassifier



def build_baseline_mlp(random_state: int = RANDOM_STATE) -> MLPClassifier:
    """
    Constrói o MLP baseline conforme o artigo.

    Arquitetura: 128 → 64 → saída
    Solver     : ADAM
    Ativação   : ReLU
    Regulariz. : L2 (alpha=0.0001)
    EarlyStopping: sim (10 épocas sem melhora)

    Returns
    -------
    MLPClassifier configurado (não treinado).
    """
    return MLPClassifier(
        hidden_layer_sizes=MLP_HIDDEN_LAYERS,
        activation=MLP_ACTIVATION,
        solver=MLP_SOLVER,
        alpha=MLP_ALPHA,
        batch_size="auto",           # mini-batch (padrão sklearn)
        learning_rate=MLP_LEARNING_RATE,
        max_iter=MLP_MAX_ITER,
        random_state=random_state,
        early_stopping=MLP_EARLY_STOP,
        validation_fraction=MLP_VAL_FRACTION,
        n_iter_no_change=MLP_N_ITER_NO_CHG,
        verbose=False,               # silencioso — o Trainer imprime o progresso
    )


def build_mlp_from_params(params: dict, random_state: int = RANDOM_STATE) -> MLPClassifier:
    """
    Constrói um MLP com hiperparâmetros arbitrários (para tuning).

    Os parâmetros fixos (activation, solver, early_stopping) são mantidos
    do baseline; apenas os variáveis do espaço de busca são substituídos.

    Parameters
    ----------
    params : dict  — hiperparâmetros do RandomizedSearchCV
    """
    return MLPClassifier(
        activation="relu",
        solver="adam",
        early_stopping=True,
        validation_fraction=MLP_VAL_FRACTION,
        random_state=random_state,
        verbose=False,
        **params,
    )


### Arquivo: `ml/training/trainer.py`


In [ ]:
"""
Treinamento do modelo MLP e validação cruzada.

SRP: este módulo gerencia exclusivamente o ciclo de treino e a
validação cruzada no conjunto de treino.

Regra do curso: validação cruzada é feita SOMENTE no conjunto de treino.
O test_set é reservado para avaliação final — nunca é usado aqui.
"""

from __future__ import annotations

import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neural_network import MLPClassifier



class ModelTrainer:
    """
    Treina o MLP baseline e avalia com validação cruzada estratificada.

    Uso:
        trainer = ModelTrainer()
        model   = trainer.train(X_train_bal, y_train_bal)
        trainer.cross_validate(X_train_bal, y_train_bal)
    """

    def __init__(
        self,
        random_state: int = RANDOM_STATE,
        cv_n_splits: int = CV_N_SPLITS,
        cv_scoring: str = CV_SCORING,
        save_plots: bool = True,
    ) -> None:
        self._random_state = random_state
        self._cv_n_splits  = cv_n_splits
        self._cv_scoring   = cv_scoring
        self._save_plots   = save_plots

    # ── API pública ────────────────────────────────────────────────────────────

    def train(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        model: MLPClassifier | None = None,
    ) -> MLPClassifier:
        """
        Treina o MLP no conjunto de treino balanceado.

        Parameters
        ----------
        X_train : np.ndarray — features escalonadas e balanceadas (pós-SMOTE)
        y_train : np.ndarray — alvo balanceado
        model   : MLPClassifier opcional; se None, usa o baseline configurado

        Returns
        -------
        MLPClassifier treinado.
        """
        if model is None:
            model = build_baseline_mlp(self._random_state)

        print("\n[ModelTrainer] Iniciando treinamento MLP baseline...")
        print(f"  Arquitetura : {model.hidden_layer_sizes}")
        print(f"  Solver      : {model.solver}")
        print(f"  Ativação    : {model.activation}")
        print(f"  max_iter    : {model.max_iter}")
        print(f"  Shape treino: {X_train.shape}")

        t0 = time.monotonic()
        model.fit(X_train, y_train)
        elapsed = time.monotonic() - t0

        print(f"\n[ModelTrainer] Treinamento concluído em {elapsed:.1f}s")
        print(f"  Épocas executadas : {model.n_iter_}")
        print(f"  Loss final        : {model.loss_:.6f}")

        if self._save_plots:
            self._plot_loss_curve(model)

        return model

    def cross_validate(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
    ) -> dict[str, tuple[float, float]]:
        """
        Avalia o baseline com StratifiedKFold no conjunto de TREINO.

        NUNCA usa X_test — a validação cruzada é exclusivamente no treino.

        Returns
        -------
        dict: métrica → (mean, std)
        """
        cv = StratifiedKFold(
            n_splits=self._cv_n_splits,
            shuffle=True,
            random_state=self._random_state,
        )

        metrics = ["accuracy", "f1", "precision", "recall"]
        results: dict[str, tuple[float, float]] = {}

        print(f"\n[ModelTrainer] Validação Cruzada ({self._cv_n_splits}-fold) — conjunto de treino:")
        print("-" * 50)

        for metric in metrics:
            scores = cross_val_score(
                build_baseline_mlp(self._random_state),
                X_train,
                y_train,
                cv=cv,
                scoring=metric,
                n_jobs=-1,
            )
            mean, std = float(scores.mean()), float(scores.std())
            results[metric] = (mean, std)
            print(f"  {metric:<12}: {mean:.4f} ± {std:.4f}")

        print("-" * 50)
        return results

    # ── Métodos privados ───────────────────────────────────────────────────────

    def _plot_loss_curve(self, model: MLPClassifier) -> None:
        """Salva a curva de convergência do modelo em outputs/."""
        try:
            OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
            fig, ax = plt.subplots(figsize=(10, 4))

            ax.plot(model.loss_curve_, label="Loss de Treino", color="steelblue")
            if hasattr(model, "validation_scores_") and model.validation_scores_:
                ax.plot(
                    model.validation_scores_,
                    label="Score de Validação Interna",
                    color="orange",
                    linestyle="--",
                )

            ax.set_xlabel("Épocas")
            ax.set_ylabel("Loss / Score")
            ax.set_title("Curva de Convergência — MLP Baseline (InSDN8)")
            ax.legend()
            plt.tight_layout()
            plt.savefig(OUTPUTS_DIR / "loss_curve_baseline.png", dpi=150, bbox_inches="tight")
            plt.close()
            print(f"[ModelTrainer] Curva de loss salva em {OUTPUTS_DIR}/loss_curve_baseline.png")
        except Exception as e:
            print(f"[ModelTrainer] Não foi possível salvar curva de loss: {e}")


### Arquivo: `ml/training/tuner.py`


In [ ]:
"""
Hyperparameter Tuning com RandomizedSearchCV.

SRP: responsável exclusivamente pela busca de hiperparâmetros.

Regra do curso:
  - Tuning é feito com StratifiedKFold NO CONJUNTO DE TREINO.
  - O test_set NÃO é tocado durante o tuning.
  - RandomizedSearchCV é mais eficiente que GridSearch para espaços grandes.
"""

from __future__ import annotations

import time

import numpy as np
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier



class HyperparameterTuner:
    """
    Busca os melhores hiperparâmetros do MLP via RandomizedSearchCV.

    Uso:
        tuner = HyperparameterTuner()
        best_model = tuner.fit(X_train_bal, y_train_bal)
        print(tuner.best_params_)
        print(tuner.best_cv_score_)
    """

    def __init__(
        self,
        n_iter: int = TUNING_N_ITER,
        param_distributions: dict | None = None,
        scoring: str = CV_SCORING,
        cv_n_splits: int = CV_N_SPLITS,
        random_state: int = RANDOM_STATE,
    ) -> None:
        self._n_iter             = n_iter
        self._param_distributions = param_distributions or TUNING_PARAM_DISTRIBUTIONS
        self._scoring            = scoring
        self._cv_n_splits        = cv_n_splits
        self._random_state       = random_state
        self._search: RandomizedSearchCV | None = None

    # ── API pública ────────────────────────────────────────────────────────────

    def fit(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
    ) -> MLPClassifier:
        """
        Executa a busca de hiperparâmetros no conjunto de TREINO.

        Parameters
        ----------
        X_train : np.ndarray — treino balanceado (pós-SMOTE)
        y_train : np.ndarray — alvo balanceado

        Returns
        -------
        best_estimator_: MLPClassifier treinado com os melhores parâmetros,
                         já fitado em todo X_train.
        """
        cv = StratifiedKFold(
            n_splits=self._cv_n_splits,
            shuffle=True,
            random_state=self._random_state,
        )

        base_mlp = MLPClassifier(
            activation="relu",
            solver="adam",
            early_stopping=True,
            validation_fraction=0.1,
            random_state=self._random_state,
            verbose=False,
        )

        self._search = RandomizedSearchCV(
            estimator=base_mlp,
            param_distributions=self._param_distributions,
            n_iter=self._n_iter,
            cv=cv,
            scoring=self._scoring,
            n_jobs=-1,
            random_state=self._random_state,
            verbose=1,
            return_train_score=True,
        )

        print(f"\n[HyperparameterTuner] Iniciando busca ({self._n_iter} combinações, "
              f"{self._cv_n_splits}-fold CV)...")
        print(f"  Scoring: {self._scoring}")
        print(f"  Espaço de busca: {list(self._param_distributions.keys())}")

        t0 = time.monotonic()
        self._search.fit(X_train, y_train)
        elapsed = time.monotonic() - t0

        print(f"\n[HyperparameterTuner] Busca concluída em {elapsed:.1f}s")
        print(f"  Melhores parâmetros:")
        for k, v in self._search.best_params_.items():
            print(f"    {k}: {v}")
        print(f"  Melhor {self._scoring} (CV): {self._search.best_score_:.4f}")

        return self._search.best_estimator_

    @property
    def best_params_(self) -> dict:
        """Melhores hiperparâmetros encontrados."""
        if self._search is None:
            raise RuntimeError("HyperparameterTuner: chame fit() primeiro.")
        return self._search.best_params_

    @property
    def best_cv_score_(self) -> float:
        """Melhor score CV encontrado."""
        if self._search is None:
            raise RuntimeError("HyperparameterTuner: chame fit() primeiro.")
        return float(self._search.best_score_)

    @property
    def cv_results_(self) -> dict:
        """Resultados completos do RandomizedSearchCV."""
        if self._search is None:
            raise RuntimeError("HyperparameterTuner: chame fit() primeiro.")
        return self._search.cv_results_


### Arquivo: `ml/evaluation/evaluator.py`


In [ ]:
"""
Avaliação completa do modelo com métricas e visualizações.

SRP: responsável exclusivamente pelo cálculo de métricas e geração de plots.

O test_set é usado UMA ÚNICA VEZ aqui — após todo o desenvolvimento.
Nunca é usado para ajustar parâmetros ou tomar decisões de modelo.

Métricas implementadas (conforme boas práticas do curso para dados desbalanceados):
  - Acurácia, Precisão, Recall, F1-Score
  - MCC (Matthews Correlation Coefficient) — melhor para desbalanceamento
  - Geometric Mean — balanceia sensibilidade e especificidade
  - ROC-AUC
"""

from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from imblearn.metrics import geometric_mean_score
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.neural_network import MLPClassifier



@dataclass
class EvaluationResult:
    """
    Resultado imutável de uma avaliação.

    Armazena todas as métricas calculadas no test_set para
    fácil comparação entre baseline e modelo otimizado.
    """

    label:     str
    accuracy:  float
    precision: float
    recall:    float
    f1:        float
    mcc:       float
    gm:        float
    roc_auc:   float
    tp: int = 0
    tn: int = 0
    fp: int = 0
    fn: int = 0

    # Texto do relatório sklearn (opcional)
    classification_report: str = field(default="", repr=False)

    def print_summary(self) -> None:
        """Exibe o resumo formatado das métricas."""
        width = 58
        print("\n" + "=" * width)
        print(f"{f'RESULTADOS — {self.label}':^{width}}")
        print("=" * width)
        print(f"  Acurácia          : {self.accuracy:.4f}  ({self.accuracy*100:.2f}%)")
        print(f"  Precisão          : {self.precision:.4f}  ({self.precision*100:.2f}%)")
        print(f"  Recall/Sensibil.  : {self.recall:.4f}  ({self.recall*100:.2f}%)")
        print(f"  F1-Score          : {self.f1:.4f}  ({self.f1*100:.2f}%)")
        print(f"  MCC               : {self.mcc:.4f}")
        print(f"  Geometric Mean    : {self.gm:.4f}")
        print(f"  ROC-AUC           : {self.roc_auc:.4f}")
        print("-" * width)
        print(f"  TP={self.tp:,}  TN={self.tn:,}  FP={self.fp:,}  FN={self.fn:,}")
        print("=" * width)


class ModelEvaluator:
    """
    Avalia um modelo MLPClassifier no test_set e gera relatórios/plots.

    Uso:
        evaluator = ModelEvaluator()
        result    = evaluator.evaluate(model, X_test_scaled, y_test, label="Baseline")
        evaluator.compare(baseline_result, optimized_result)
    """

    def __init__(self, save_plots: bool = True) -> None:
        self._save_plots = save_plots
        OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

    # ── API pública ────────────────────────────────────────────────────────────

    def evaluate(
        self,
        model: MLPClassifier,
        X_test: np.ndarray,
        y_test: np.ndarray,
        label: str = "Modelo",
    ) -> EvaluationResult:
        """
        Avalia o modelo no test_set.

        O test_set representa dados "do mundo real" — nunca foi visto
        durante o treinamento ou tuning.

        Parameters
        ----------
        model   : MLPClassifier treinado
        X_test  : np.ndarray — features do teste (escalonadas e selecionadas)
        y_test  : np.ndarray — alvo real do teste
        label   : str        — nome do modelo para exibição

        Returns
        -------
        EvaluationResult com todas as métricas calculadas.
        """
        y_pred       = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]

        cm           = confusion_matrix(y_test, y_pred)
        tn, fp, fn, tp = cm.ravel()

        result = EvaluationResult(
            label=label,
            accuracy=float(accuracy_score(y_test, y_pred)),
            precision=float(precision_score(y_test, y_pred, zero_division=0)),
            recall=float(recall_score(y_test, y_pred, zero_division=0)),
            f1=float(f1_score(y_test, y_pred, zero_division=0)),
            mcc=float(matthews_corrcoef(y_test, y_pred)),
            gm=float(geometric_mean_score(y_test, y_pred)),
            roc_auc=float(roc_auc_score(y_test, y_pred_proba)),
            tp=int(tp),
            tn=int(tn),
            fp=int(fp),
            fn=int(fn),
            classification_report=classification_report(
                y_test, y_pred,
                target_names=["Benigno (0)", "Ataque DDoS (1)"],
            ),
        )

        result.print_summary()

        print("\nRelatório de Classificação:")
        print(result.classification_report)

        if self._save_plots:
            slug = label.lower().replace(" ", "_").replace("(", "").replace(")", "")
            self._plot_confusion_matrix(cm, label, slug)
            self._plot_roc_curve(y_test, y_pred_proba, result.roc_auc, label, slug)

        return result

    def compare(
        self,
        baseline: EvaluationResult,
        optimized: EvaluationResult,
    ) -> None:
        """
        Exibe tabela comparativa: Baseline vs. Otimizado vs. Artigo.
        """
        print("\n" + "=" * 65)
        print(f"{'COMPARAÇÃO DE MODELOS':^65}")
        print("=" * 65)
        print(f"{'Métrica':<22} {'Baseline':>10} {'Otimizado':>10} {'Artigo':>10}")
        print("-" * 65)

        metrics_map = {
            "Acurácia (%)":   ("accuracy",  "%"),
            "Precisão (%)":   ("precision", "%"),
            "Recall (%)":     ("recall",    "%"),
            "F1-Score (%)":   ("f1",        "%"),
            "MCC":            ("mcc",       "raw"),
            "Geometric Mean": ("gm",        "raw"),
            "ROC-AUC":        ("roc_auc",   "raw"),
        }

        for name, (attr, fmt) in metrics_map.items():
            b_val = getattr(baseline, attr)
            o_val = getattr(optimized, attr)
            paper_key = attr if attr in PAPER_METRICS else None

            if fmt == "%":
                b_str = f"{b_val*100:>10.2f}"
                o_str = f"{o_val*100:>10.2f}"
                paper = f"{PAPER_METRICS[paper_key]:>10.2f}" if paper_key else f"{'—':>10}"
            else:
                b_str = f"{b_val:>10.4f}"
                o_str = f"{o_val:>10.4f}"
                paper = f"{'—':>10}"

            print(f"{name:<22}{b_str}{o_str}{paper}")

        print("=" * 65)

        # Ganho do tuning
        f1_gain = (optimized.f1 - baseline.f1) * 100
        print(f"\n  Ganho de F1 (tuning): {f1_gain:+.2f} pp")

        # Distância ao artigo
        paper_f1 = PAPER_METRICS.get("f1")
        if paper_f1:
            dist = paper_f1 - optimized.f1 * 100
            print(f"  Distância ao artigo (F1): {dist:.2f} pp")

    # ── Métodos privados ───────────────────────────────────────────────────────

    def _plot_confusion_matrix(
        self,
        cm: np.ndarray,
        label: str,
        slug: str,
    ) -> None:
        try:
            fig, ax = plt.subplots(figsize=(7, 5))
            ConfusionMatrixDisplay(
                confusion_matrix=cm,
                display_labels=["Benigno", "Ataque DDoS"],
            ).plot(values_format=".0f", ax=ax, cmap="Blues")
            ax.set_title(f"Matriz de Confusão — {label} (InSDN8)")
            plt.tight_layout()
            path = OUTPUTS_DIR / f"confusion_matrix_{slug}.png"
            plt.savefig(path, dpi=150, bbox_inches="tight")
            plt.close()
            print(f"[ModelEvaluator] Matriz de confusão salva: {path.name}")
        except Exception as e:
            print(f"[ModelEvaluator] Erro ao salvar matriz de confusão: {e}")

    def _plot_roc_curve(
        self,
        y_test: np.ndarray,
        y_pred_proba: np.ndarray,
        auc: float,
        label: str,
        slug: str,
    ) -> None:
        try:
            fig, ax = plt.subplots(figsize=(7, 5))
            RocCurveDisplay.from_predictions(
                y_test,
                y_pred_proba,
                name=f"{label} (AUC={auc:.4f})",
                ax=ax,
            )
            ax.plot([0, 1], [0, 1], "k--", label="Aleatório")
            ax.set_title(f"Curva ROC — {label} (InSDN8)")
            ax.legend()
            plt.tight_layout()
            path = OUTPUTS_DIR / f"roc_curve_{slug}.png"
            plt.savefig(path, dpi=150, bbox_inches="tight")
            plt.close()
            print(f"[ModelEvaluator] Curva ROC salva: {path.name}")
        except Exception as e:
            print(f"[ModelEvaluator] Erro ao salvar curva ROC: {e}")


### Arquivo: `ml/inference/predictor.py`


In [ ]:
"""
Inferência em produção: DDoSPredictor.

SRP: responsável exclusivamente pela aplicação do pipeline de transformações
e predição em novos dados (tráfego de rede capturado em tempo real).

Garante que as mesmas transformações usadas no treino sejam aplicadas
na mesma ordem durante a inferência — usando os MESMOS objetos fitados.

Pipeline de inferência (ordem obrigatória):
  1. replace Inf → NaN
  2. imputer.transform()          (mediana do treino)
  3. variance_filter.transform()  (remove colunas de variância zero)
  4. selecionar features          (top-N do SHAP)
  5. scaler.transform()           (z-score do treino)
  6. model.predict() / predict_proba()
"""

from __future__ import annotations

import numpy as np
import pandas as pd



class DDoSPredictor:
    """
    Aplica o pipeline completo de inferência a novos dados de rede.

    Uso:
        predictor = DDoSPredictor()
        predictor.load()
        predictions = predictor.predict(df_novos_fluxos)
        probabilities = predictor.predict_proba(df_novos_fluxos)

    O DataFrame de entrada deve conter as mesmas colunas que o dataset
    de treinamento (sem a coluna Label).
    """

    def __init__(self, models_dir: str | None = None) -> None:
        self._io        = ModelIO(*([models_dir] if models_dir else []))
        self._artifacts: PipelineArtifacts | None = None

    # ── API pública ────────────────────────────────────────────────────────────

    def load(self) -> "DDoSPredictor":
        """
        Carrega todos os artefatos do pipeline salvo.

        Returns
        -------
        self (fluent interface).
        """
        self._artifacts = self._io.load()
        print("[DDoSPredictor] Pipeline carregado com sucesso.")
        print(f"  Features esperadas ({len(self._artifacts.selected_features)}): "
              f"{self._artifacts.selected_features}")
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        """
        Prediz a classe de cada fluxo de rede.

        Parameters
        ----------
        X : pd.DataFrame — features brutas (sem o alvo)
                           deve conter as colunas originais do dataset

        Returns
        -------
        np.ndarray de inteiros: 0 = Benigno, 1 = Ataque DDoS
        """
        X_processed = self._preprocess(X)
        return self._artifacts.model.predict(X_processed)

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        """
        Retorna as probabilidades de cada classe.

        Returns
        -------
        np.ndarray de shape (n_samples, 2):
          coluna 0 = P(Benigno), coluna 1 = P(Ataque DDoS)
        """
        X_processed = self._preprocess(X)
        return self._artifacts.model.predict_proba(X_processed)

    def predict_with_confidence(
        self, X: pd.DataFrame, threshold: float = 0.5
    ) -> pd.DataFrame:
        """
        Prediz classe + probabilidade de ataque + flag de alta confiança.

        Parameters
        ----------
        threshold : float — limiar de probabilidade para classificar como ataque
                            (padrão 0.5; diminuir aumenta recall, aumenta falsos positivos)

        Returns
        -------
        pd.DataFrame com colunas:
          prediction  : int   — 0 ou 1
          prob_attack : float — probabilidade de ser ataque DDoS
          high_conf   : bool  — True se prob_attack > 0.9
        """
        proba       = self.predict_proba(X)
        prob_attack = proba[:, 1]
        prediction  = (prob_attack >= threshold).astype(int)

        return pd.DataFrame({
            "prediction":  prediction,
            "prob_attack": prob_attack,
            "high_conf":   prob_attack > 0.9,
        })

    # ── Métodos privados ───────────────────────────────────────────────────────

    def _preprocess(self, X: pd.DataFrame) -> np.ndarray:
        """
        Aplica o pipeline de transformações na mesma ordem do treino.

        Raises
        ------
        RuntimeError se load() não foi chamado.
        ValueError   se colunas necessárias estiverem ausentes.
        """
        if self._artifacts is None:
            raise RuntimeError(
                "DDoSPredictor: chame load() antes de predict()."
            )

        a = self._artifacts

        # 1. Infinitos → NaN
        X_clean = X.replace([np.inf, -np.inf], np.nan)

        # 2. Imputação (transform apenas — sem re-fit)
        X_imputed = pd.DataFrame(
            a.imputer.transform(X_clean),
            columns=X_clean.columns,
        )

        # 3. VarianceThreshold (transform apenas)
        surviving_cols = X_imputed.columns[a.variance_filter.get_support()].tolist()
        X_var = pd.DataFrame(
            a.variance_filter.transform(X_imputed),
            columns=surviving_cols,
        )

        # 4. Selecionar features (mesmas do treino)
        missing_features = [f for f in a.selected_features if f not in X_var.columns]
        if missing_features:
            raise ValueError(
                f"DDoSPredictor: features ausentes nos dados de entrada: {missing_features}\n"
                f"Features esperadas: {a.selected_features}"
            )
        X_sel = X_var[a.selected_features]

        # 5. Escalonamento (transform apenas — sem re-fit)
        X_scaled = a.scaler.transform(X_sel)

        return X_scaled


### Arquivo: `ml/persistence/model_io.py`


In [ ]:
"""
Persistência de artefatos do pipeline ML.

SRP: responsável exclusivamente por salvar e carregar todos os objetos
fitados durante o treinamento.

Boas práticas: salvar TODOS os transformadores junto ao modelo.
Em produção, os novos dados devem passar pelas mesmas transformações,
usando os MESMOS objetos (com os mesmos parâmetros aprendidos no treino).

Artefatos salvos:
  mlp_ddos_insdn.joblib    — modelo MLP treinado
  imputer.joblib           — SimpleImputer (fit no treino)
  variance_filter.joblib   — VarianceThreshold (fit no treino)
  scaler.joblib            — StandardScaler (fit no treino)
  selected_features.joblib — lista de features selecionadas pelo SHAP
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import joblib
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler



@dataclass
class PipelineArtifacts:
    """
    Conjunto de artefatos que compõem o pipeline de inferência.

    Todos os campos devem estar fitados com os dados de TREINO antes
    de serem salvos e usados em produção.
    """

    model:             MLPClassifier
    imputer:           SimpleImputer
    variance_filter:   VarianceThreshold
    scaler:            StandardScaler
    selected_features: list[str]


class ModelIO:
    """
    Salva e carrega o pipeline completo de detecção de DDoS.

    Uso (treino):
        io = ModelIO()
        io.save(artifacts)

    Uso (produção):
        io       = ModelIO()
        pipeline = io.load()
        preds    = pipeline.model.predict(scaler.transform(...))
    """

    _FILENAMES = {
        "model":             "mlp_ddos_insdn.joblib",
        "imputer":           "imputer.joblib",
        "variance_filter":   "variance_filter.joblib",
        "scaler":            "scaler.joblib",
        "selected_features": "selected_features.joblib",
    }

    def __init__(self, models_dir: Path | str = MODELS_DIR) -> None:
        self._dir = Path(models_dir)

    # ── API pública ────────────────────────────────────────────────────────────

    def save(self, artifacts: PipelineArtifacts) -> None:
        """
        Salva todos os artefatos do pipeline em models/.

        Cria o diretório se não existir.
        """
        self._dir.mkdir(parents=True, exist_ok=True)

        pairs = [
            ("model",             artifacts.model),
            ("imputer",           artifacts.imputer),
            ("variance_filter",   artifacts.variance_filter),
            ("scaler",            artifacts.scaler),
            ("selected_features", artifacts.selected_features),
        ]

        print(f"\n[ModelIO] Salvando artefatos em {self._dir}/")
        for key, obj in pairs:
            path = self._dir / self._FILENAMES[key]
            with open(path, "wb") as f:
                joblib.dump(obj, f)
            print(f"  ✓ {self._FILENAMES[key]}")

        print(f"[ModelIO] {len(pairs)} artefatos salvos com sucesso.")
        print("\n  Estrutura salva:")
        print(f"  {self._dir}/")
        for fname in self._FILENAMES.values():
            print(f"    ├── {fname}")

    def load(self) -> PipelineArtifacts:
        """
        Carrega todos os artefatos do pipeline a partir de models/.

        Returns
        -------
        PipelineArtifacts com todos os objetos fitados prontos para inferência.

        Raises
        ------
        FileNotFoundError se qualquer artefato estiver ausente.
        """
        print(f"[ModelIO] Carregando artefatos de {self._dir}/")

        loaded = {}
        for key, fname in self._FILENAMES.items():
            path = self._dir / fname
            if not path.exists():
                raise FileNotFoundError(
                    f"Artefato não encontrado: {path}\n"
                    f"Execute o pipeline de treinamento primeiro."
                )
            with open(path, "rb") as f:
                loaded[key] = joblib.load(f)
            print(f"  ✓ {fname}")

        return PipelineArtifacts(
            model=loaded["model"],
            imputer=loaded["imputer"],
            variance_filter=loaded["variance_filter"],
            scaler=loaded["scaler"],
            selected_features=loaded["selected_features"],
        )

    def exists(self) -> bool:
        """Verifica se todos os artefatos já foram salvos."""
        return all(
            (self._dir / fname).exists()
            for fname in self._FILENAMES.values()
        )


### Arquivo: `ml/pipeline.py`


In [ ]:
"""
Pipeline principal de detecção de DDoS em SDN com MLP.

Orquestra todas as etapas do workflow conforme as boas práticas do curso
(Thaís Gaudencio, UFPB/LUMO) e o plano de implementação baseado em
Mehmood et al. (PLoS ONE, 2025).

Ordem rigorosa do pipeline (nenhuma etapa pode ser invertida):
  1.  Configurações e reprodutibilidade
  2.  Carregamento do dataset InSDN8
  3.  EDA exploratória (sem modificar os dados)
  4.  ⚠️  SPLIT treino/teste ANTES de qualquer transformação  ⚠️
  5.  Limpeza (duplicatas, Inf → NaN, imputação) — somente no treino
  6.  Seleção de features (VarianceThreshold + SHAP) — somente no treino
  7.  Escalonamento (StandardScaler.fit no treino) — transform no teste
  8.  Balanceamento SMOTE — somente no treino
  9.  Treinamento MLP baseline + validação cruzada
  10. Avaliação baseline no test_set
  11. Hyperparameter Tuning (RandomizedSearchCV no treino)
  12. Avaliação final do modelo otimizado no test_set
  13. Comparação baseline vs. otimizado vs. artigo
  14. Salvamento de todos os artefatos

Avaliação do dataset insdn8_ddos_binary_0n1d.csv:
  ✓ Compatível com o plano — subconjunto do InSDN com 8 features pré-selecionadas
  ✓ Label já binarizada (0=Benigno, 1=Ataque DDoS)
  ✓ ~190.366 instâncias — volume adequado para MLP
  ⚠ Fortemente desbalanceado (maioria classe 1) — SMOTE obrigatório
  ⚠ Apenas 8 features (vs. 84 do InSDN original) — seleção SHAP será feita,
    mas sem redução de dimensionalidade significativa esperada
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split



# ── Reprodutibilidade global ───────────────────────────────────────────────────
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 7000)
pd.set_option("display.max_rows", 90000)


def run_pipeline(
    run_tuning: bool = True,
    run_eda: bool = True,
    verbose: bool = True,
    run_id: str | None = None,
) -> None:
    """
    Executa o pipeline completo de treinamento e avaliação.

    Parameters
    ----------
    run_tuning : bool — se True, executa o hyperparameter tuning (mais lento)
    run_eda    : bool — se True, exibe a análise exploratória textual
    verbose    : bool — nível de verbosidade geral
    """
    OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    print("=" * 60)
    print("  Pipeline: Detecção de DDoS em SDN com MLP")
    print("  Dataset : InSDN8 (insdn8_ddos_binary_0n1d.csv)")
    print("  Baseline: Mehmood et al. (PLoS ONE, 2025)")
    print("=" * 60)

    # ── Etapa 1: Configurações ─────────────────────────────────────────────────
    print(f"\n[1/14] Configurações")
    print(f"  RANDOM_STATE : {RANDOM_STATE}")
    print(f"  TEST_SIZE    : {TEST_SIZE} (70/30)")
    print(f"  TARGET_COL   : {TARGET_COL}")

    # ── Etapa 2: Carregamento ──────────────────────────────────────────────────
    print(f"\n[2/14] Carregando dataset...")
    loader = InSDNLoader()
    X, y   = loader.load()

    # ── Etapa 3: EDA ──────────────────────────────────────────────────────────
    if run_eda:
        print(f"\n[3/14] EDA (sem modificar os dados)...")
        loader.describe()
    else:
        print(f"\n[3/14] EDA ignorada (run_eda=False)")

    # ── Etapa 4: SPLIT (ANTES de qualquer transformação) ──────────────────────
    print(f"\n[4/14] Split estratificado {int((1-TEST_SIZE)*100)}/{int(TEST_SIZE*100)}...")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True,
        stratify=y,       # OBRIGATÓRIO: preserva proporção de classes
    )
    print(f"  Train: {X_train.shape} | Test: {X_test.shape}")
    print(f"  Distribuição treino:\n  {y_train.value_counts(normalize=True).mul(100).round(2).to_dict()}")
    print(f"  Distribuição teste :\n  {y_test.value_counts(normalize=True).mul(100).round(2).to_dict()}")

    # Resetar índices para evitar problemas de alinhamento nas etapas seguintes
    X_train = X_train.reset_index(drop=True)
    X_test  = X_test.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)
    y_test  = y_test.reset_index(drop=True)

    # ── Etapa 5: Limpeza (somente no treino) ──────────────────────────────────
    print(f"\n[5/14] Limpeza e preparação (somente no treino)...")
    cleaner          = DataCleaner()
    X_train, y_train = cleaner.fit_transform(X_train, y_train)
    X_test           = cleaner.transform(X_test)

    # ── Etapa 6: Seleção de features (somente no treino) ─────────────────────
    print(f"\n[6/14] Seleção de features (VarianceThreshold + SHAP)...")
    selector    = FeatureSelector()
    X_train_sel = selector.fit_transform(X_train, y_train)
    X_test_sel  = selector.transform(X_test)

    print(f"\n  Features selecionadas: {selector.selected_features}")
    print(f"  Shape treino selecionado: {X_train_sel.shape}")
    print(f"  Shape teste  selecionado: {X_test_sel.shape}")

    # ── Etapa 7: Escalonamento (fit no treino) ────────────────────────────────
    print(f"\n[7/14] Escalonamento (StandardScaler.fit no treino)...")
    scaler         = FeatureScaler()
    X_train_scaled = scaler.fit_transform(X_train_sel)
    X_test_scaled  = scaler.transform(X_test_sel)

    # ── Etapa 8: Balanceamento SMOTE (somente no treino) ─────────────────────
    print(f"\n[8/14] Balanceamento SMOTE (somente no treino)...")
    balancer             = ClassBalancer()
    X_train_bal, y_train_bal = balancer.fit_resample(X_train_scaled, y_train)

    # ── Etapa 9: Treinamento MLP baseline + CV ────────────────────────────────
    print(f"\n[9/14] Treinamento MLP baseline...")
    trainer       = ModelTrainer(save_plots=True)
    model_baseline = trainer.train(X_train_bal, y_train_bal)

    print(f"\n[9b/14] Validação cruzada no treino...")
    cv_results = trainer.cross_validate(X_train_bal, y_train_bal)

    # ── Etapa 10: Avaliação baseline no test_set ──────────────────────────────
    print(f"\n[10/14] Avaliação BASELINE no test_set...")
    evaluator      = ModelEvaluator(save_plots=True)
    result_baseline = evaluator.evaluate(
        model_baseline,
        X_test_scaled,
        y_test.values,
        label="MLP Baseline",
    )

    # ── Etapa 11: Hyperparameter Tuning ───────────────────────────────────────
    tuner: HyperparameterTuner | None = None
    if run_tuning:
        print(f"\n[11/14] Hyperparameter Tuning (RandomizedSearchCV no treino)...")
        tuner      = HyperparameterTuner()
        model_best = tuner.fit(X_train_bal, y_train_bal)
    else:
        print(f"\n[11/14] Tuning ignorado (run_tuning=False) — usando baseline como modelo final.")
        model_best = model_baseline

    # ── Etapa 12: Avaliação final no test_set ────────────────────────────────
    print(f"\n[12/14] Avaliação FINAL (modelo otimizado) no test_set...")
    result_optimized = evaluator.evaluate(
        model_best,
        X_test_scaled,
        y_test.values,
        label="MLP Otimizado",
    )

    # ── Etapa 13: Comparação ─────────────────────────────────────────────────
    print(f"\n[13/14] Comparação: Baseline vs. Otimizado vs. Artigo")
    evaluator.compare(result_baseline, result_optimized)

    # ── Etapa 14: Salvamento (artefatos + métricas) ───────────────────────────
    print(f"\n[14/14] Salvando artefatos e métricas...")
    artifacts = PipelineArtifacts(
        model=model_best,
        imputer=cleaner.imputer,
        variance_filter=selector.variance_filter,
        scaler=scaler.scaler,
        selected_features=selector.selected_features,
    )
    ModelIO().save(artifacts)

    # Registrar métricas no histórico persistente
    from datetime import datetime
    logger    = MetricsLogger()
    n_dupes   = 133256 - len(X_train)   # duplicatas removidas no treino
    ds_info   = {
        "n_raw":               190366,
        "n_train_after_clean": len(X_train),
        "n_test":              len(X_test),
        "n_duplicates_removed": n_dupes,
        "n_features":          len(selector.selected_features),
    }
    baseline_params = {
        "hidden_layer_sizes": str((128, 64)),
        "alpha": 0.0001,
        "solver": "adam",
        "activation": "relu",
        "tuned": False,
    }
    ts_suffix = datetime.now().strftime("%Y%m%d_%H%M")
    logger.log(result_baseline, run_id=run_id or f"baseline_{ts_suffix}",
               params=baseline_params, dataset_info=ds_info)

    if run_tuning and tuner is not None:
        tuned_params = {**baseline_params, **tuner.best_params_, "tuned": True}
        logger.log(result_optimized, run_id=f"tuned_{ts_suffix}",
                   params=tuned_params, dataset_info=ds_info)

    logger.to_csv()
    logger.print_summary()

    print("\n" + "=" * 60)
    print("  Pipeline concluído com sucesso!")
    print(f"  Modelos  : models/")
    print(f"  Outputs  : outputs/")
    print(f"  Métricas : outputs/metrics_history.json")
    print("=" * 60)


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(
        description="Pipeline de detecção de DDoS em SDN com MLP (InSDN8)"
    )
    parser.add_argument(
        "--no-tuning",
        action="store_true",
        help="Pular o hyperparameter tuning (mais rápido, apenas baseline)",
    )
    parser.add_argument(
        "--no-eda",
        action="store_true",
        help="Pular a análise exploratória textual",
    )
    parser.add_argument(
        "--run-id",
        type=str,
        default=None,
        help="Identificador da run para o histórico de métricas (ex: 'experimento_v2')",
    )
    args = parser.parse_args()

    run_pipeline(
        run_tuning=not args.no_tuning,
        run_eda=not args.no_eda,
        run_id=args.run_id,
    )


### Execução do Pipeline Binário
Descomente a célula abaixo para executar:


In [ ]:
# run_pipeline(run_tuning=False, run_eda=True, run_id='notebook_binary')

## Parte 2: Pipeline de Classificação Triclasse


### Arquivo: `ml/triclass/config.py`


In [ ]:
"""
Configurações do pipeline triclasse de detecção de DDoS em SDN.

Dataset: InSDN (Elsayed et al., IEEE Access, 2020)
Classes: 0=Benigno | 1=Ataque Externo | 2=Zumbi Interno

Todas as constantes ficam aqui (SRP) — qualquer ajuste afeta apenas este arquivo.
"""

from pathlib import Path

# ── Reprodutibilidade ──────────────────────────────────────────────────────────
RANDOM_STATE: int = 42

# ── Caminhos ───────────────────────────────────────────────────────────────────
PROJECT_ROOT        = Path(__file__).resolve().parent.parent.parent
INSDN_DIR           = PROJECT_ROOT / "dataset" / "InSDN_DatasetCSV"
NORMAL_CSV          = INSDN_DIR / "Normal_data.csv"
OVS_CSV             = INSDN_DIR / "OVS.csv"
META_CSV            = INSDN_DIR / "metasploitable-2.csv"
MODELS_TRICLASS_DIR = PROJECT_ROOT / "models_triclass"
OUTPUTS_TRICLASS    = PROJECT_ROOT / "outputs_triclass"

# ── Split ──────────────────────────────────────────────────────────────────────
# 70/30 — padrão do curso (Thaís Gaudencio, UFPB/LUMO)
TEST_SIZE: float = 0.30

# ── Renomeação de colunas InSDN → padrão do plano ─────────────────────────────
# InSDN usa nomenclatura CICFlowMeter diferente. Renomear antes de qualquer op.
RENAME_MAP: dict[str, str] = {
    "Tot Fwd Pkts":    "Total Fwd Packets",
    "Tot Bwd Pkts":    "Total Backward Packets",
    "TotLen Fwd Pkts": "Total Length of Fwd Packets",
    "TotLen Bwd Pkts": "Total Length of Bwd Packets",
    "Pkt Len Std":     "Packet Length Std",
    "Fwd Pkts/s":      "Fwd Packets/s",
    "Bwd Pkts/s":      "Bwd Packets/s",
}

# ── Labels originais do InSDN usados por classe ────────────────────────────────
LABEL_BENIGN   = "Normal"
LABELS_EXTERNAL = {"DDoS"}               # somente com burst=True
LABELS_INTERNAL = {"BOTNET", "DoS"}      # BOTNET sempre; DoS somente sem burst
LABELS_DISCARD  = {"Probe", "BFA", "Web-Attack", "U2R", "UDP-lag"}

# ── Heurística de burst (seção 5.1 do plano) ──────────────────────────────────
# BOTNET tem Flow Duration ~31.000 µs >> 500 µs → is_burst() == False (correto)
# DDoS Hping3 tem duration de 1-19 µs          → is_burst() == True  (correto)
BURST_PKT_LEN_STD_MAX: float = 1.0     # Packet Length Std ≤ 1.0
BURST_FLOW_DURATION_MAX: float = 500.0  # Flow Duration < 500 µs

# ── Pré-processamento ──────────────────────────────────────────────────────────
IMPUTER_STRATEGY: str = "median"
VARIANCE_THRESHOLD: float = 0.01

# ── Features comportamentais ───────────────────────────────────────────────────
BEHAVIORAL_FEATURES: list[str] = [
    "asymmetry_pkts",
    "asymmetry_bytes",
    "pkt_rate",
    "pkt_uniformity",
    "log_duration",
    "fwd_active_ratio",
]

# ── Nomes das classes para exibição ───────────────────────────────────────────
CLASS_NAMES: dict[int, str] = {
    0: "Benigno",
    1: "Externo",
    2: "Zumbi Interno",
}

# ── SMOTE conservador (seção 8.7 do plano) ────────────────────────────────────
# Classe 2 vai para min(5x original, tamanho da Classe 0)
SMOTE_MAX_FACTOR_CLS2: int = 5
# Undersample classe 1 se for > 2x a classe 0
SMOTE_UNDERSAMPLE_RATIO_CLS1: float = 1.5

# ── Random Forest ─────────────────────────────────────────────────────────────
RF_N_ESTIMATORS: int = 200
RF_CLASS_WEIGHT: str = "balanced"

RF_TUNING_PARAM_DIST: dict = {
    "n_estimators":      [100, 200, 300, 500],
    "max_depth":         [10, 20, 30, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf":  [1, 2, 4],
    "max_features":      ["sqrt", "log2"],
}
RF_TUNING_N_ITER: int = 30

# ── MLP (comparativo) ─────────────────────────────────────────────────────────
MLP_HIDDEN_LAYERS: tuple = (128, 64)
MLP_ACTIVATION:    str   = "relu"
MLP_SOLVER:        str   = "adam"
MLP_MAX_ITER:      int   = 300
MLP_EARLY_STOP:    bool  = True
MLP_N_ITER_NO_CHG: int   = 15

# ── Validação cruzada ──────────────────────────────────────────────────────────
CV_N_SPLITS: int = 10
CV_SCORING:  str = "f1_macro"

# ── Validação semântica BOTNET ─────────────────────────────────────────────────
# Recall mínimo esperado para BOTNET no teste (seção 8.11 do plano)
BOTNET_MIN_RECALL: float = 0.80

# ── Permutation Importance / Ablation Study ────────────────────────────────────
# Features suspeitas de ser "atalhos de identidade": o modelo pode usar
# porta/protocolo como proxy do tipo de tráfego em vez de aprender padrões
# comportamentais reais. Em produção, essas features podem não estar disponíveis
# ou ter distribuição diferente (ex: porta 80 pode ser legítimo ou ataque).
IDENTITY_FEATURES: list[str] = [
    "Dst Port",
    "Src Port",
    "Protocol",
    "Bwd Header Len",
    "Init Bwd Win Byts",
    "Fwd Header Len",
]
# Número de repetições da permutação (30 é o mínimo para estimativa estável)
PERMUTATION_N_REPEATS: int = 30


### Arquivo: `ml/triclass/data/loader.py`


In [ ]:
"""
Carregamento dos três arquivos InSDN para o pipeline triclasse.

SRP: carrega, concatena e valida os CSVs — sem nenhuma transformação.
     Toda transformação (limpeza, features, split) ocorre downstream.

Avaliação dos arquivos:
  Normal_data.csv    — 68.424 linhas, label='Normal' (Classe 0 completa)
  OVS.csv            — 138.722 linhas, labels: DDoS, DoS, Probe, BFA,
                       Web-Attack, BOTNET
  metasploitable-2.csv — 136.743 linhas, labels: DDoS, Probe, DoS, BFA, U2R

Apenas Normal, DDoS, DoS e BOTNET são usados — o resto é descartado
na etapa de labeling (TriclassLabeler).
"""

from __future__ import annotations

import numpy as np
import pandas as pd


# Colunas necessárias para as features do plano (após renomeação)
_REQUIRED_COLS = {
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Flow Duration",
    "Packet Length Std",
    "Fwd Act Data Pkts",
    "Label",
}


class InSDNTriclassLoader:
    """
    Carrega os três arquivos do InSDN e retorna DataFrame concatenado.

    Uso:
        loader = InSDNTriclassLoader()
        df = loader.load()      # DataFrame bruto com Label original
        loader.describe(df)     # EDA textual (sem modificar df)
    """

    def __init__(
        self,
        normal_path=NORMAL_CSV,
        ovs_path=OVS_CSV,
        meta_path=META_CSV,
    ) -> None:
        self._paths = {
            "Normal_data": normal_path,
            "OVS":         ovs_path,
            "metasploitable": meta_path,
        }

    # ── API pública ────────────────────────────────────────────────────────────

    def load(self) -> pd.DataFrame:
        """
        Carrega, concatena e renomeia colunas dos três arquivos.

        Returns
        -------
        pd.DataFrame com Label original preservado e colunas padronizadas.

        Raises
        ------
        FileNotFoundError  se algum CSV não existir.
        ValueError         se colunas obrigatórias estiverem ausentes.
        """
        frames = []
        for name, path in self._paths.items():
            path = path if hasattr(path, "exists") else __import__("pathlib").Path(path)
            if not path.exists():
                raise FileNotFoundError(
                    f"[InSDNTriclassLoader] Arquivo não encontrado: {path}\n"
                    f"Verifique ml/triclass/config.py → INSDN_DIR."
                )
            df = pd.read_csv(path, low_memory=False)
            df.columns = df.columns.str.strip()
            df["_source"] = name
            frames.append(df)
            print(f"[InSDNTriclassLoader] {name}: {df.shape[0]:,} linhas")

        data = pd.concat(frames, ignore_index=True)

        # Renomear colunas para padrão do plano (seção 2.2)
        data = data.rename(columns={
            k: v for k, v in RENAME_MAP.items() if k in data.columns
        })

        self._validate_columns(data)
        print(f"[InSDNTriclassLoader] Total concatenado: {data.shape[0]:,} linhas, "
              f"{data.shape[1]} colunas")
        return data

    def describe(self, data: pd.DataFrame) -> None:
        """
        EDA textual sobre o DataFrame bruto.
        Não modifica os dados — chamada segura antes do split.
        """
        print("\n" + "=" * 65)
        print("  EDA — InSDN Triclasse (dados brutos, pré-split)")
        print("=" * 65)
        print(f"\nShape       : {data.shape}")
        print(f"Memória     : {data.memory_usage(deep=True).sum() / 1e6:.1f} MB")

        print("\n── Distribuição de Labels originais ──")
        vc = data["Label"].value_counts()
        for lbl, cnt in vc.items():
            pct = 100 * cnt / len(data)
            print(f"  {lbl:<20}: {cnt:>8,}  ({pct:.1f}%)")

        print("\n── Missing values ──")
        missing = data.isnull().sum()
        if missing.any():
            print(missing[missing > 0].to_string())
        else:
            print("  Nenhum missing value.")

        print("\n── Infinitos ──")
        num = data.select_dtypes(include=np.number)
        inf_count = np.isinf(num).sum()
        inf_cols = inf_count[inf_count > 0]
        if not inf_cols.empty:
            print(inf_cols.to_string())
        else:
            print("  Nenhum valor infinito.")

        print("\n── Duplicatas ──")
        print(f"  Linhas duplicadas: {data.duplicated().sum():,}")

        print("\n── Stats das features-chave ──")
        key_cols = [c for c in [
            "Flow Duration", "Packet Length Std",
            "Total Fwd Packets", "Total Backward Packets",
            "Fwd Act Data Pkts",
        ] if c in data.columns]
        if key_cols:
            print(data[key_cols].describe().round(4).to_string())
        print("=" * 65 + "\n")

    # ── Métodos privados ───────────────────────────────────────────────────────

    def _validate_columns(self, data: pd.DataFrame) -> None:
        missing = _REQUIRED_COLS - set(data.columns)
        if missing:
            raise ValueError(
                f"[InSDNTriclassLoader] Colunas obrigatórias ausentes após renomeação:\n"
                f"  {sorted(missing)}\n"
                f"  Ajuste RENAME_MAP em ml/triclass/config.py."
            )


### Arquivo: `ml/triclass/preprocessing/feature_engineer.py`


In [ ]:
"""
Engenharia de features comportamentais para o pipeline triclasse.

As seis features derivadas substituem o TTL (indisponível no InSDN) como
diferenciadores de comportamento entre as três classes.

SRP: apenas computa features derivadas — sem fit, sem estado, sem leakage.
     É seguro aplicar em treino e teste de forma idêntica pois não usa
     nenhuma estatística dos dados (operações puramente matemáticas).

Referência: plano_triclasse_insdn_v4.md, Seção 5.3
"""

from __future__ import annotations

import numpy as np
import pandas as pd


# Nomes das colunas-fonte (após RENAME_MAP)
_FWD   = "Total Fwd Packets"
_BWD   = "Total Backward Packets"
_FWDB  = "Total Length of Fwd Packets"
_BWDB  = "Total Length of Bwd Packets"
_DUR   = "Flow Duration"
_STD   = "Packet Length Std"
_ACT   = "Fwd Act Data Pkts"

_EPS   = 1e-9  # evitar divisão por zero


def computar_features_comportamentais(df: pd.DataFrame) -> pd.DataFrame:
    """
    Computa as seis features comportamentais e as adiciona ao DataFrame.

    Feature 1 — asymmetry_pkts:
        fwd / (fwd + bwd + ε)
        ~1.0 = tráfego unidirecional (ataque).
        ~0.5 = tráfego bidirecional (legítimo).

    Feature 2 — asymmetry_bytes:
        fwd_bytes / (fwd_bytes + bwd_bytes + ε)
        Amplificação DNS/NTP tem assimetria inversa (~0) por resposta > req.

    Feature 3 — pkt_rate:
        (fwd + bwd) / (duration + ε)
        Alta taxa = ataque volumétrico.

    Feature 4 — pkt_uniformity:
        1 / (Packet_Length_Std + 1)
        Alta (~1) = gerado por script (burst).
        Baixa (~0) = tráfego humano variado.

    Feature 5 — log_duration:
        log1p(Flow_Duration)
        Normaliza distribuição de cauda longa (Aula 5 — log1p em assimétricos).
        BOTNET ~31.000 µs >> DDoS ~1-19 µs → diferenciador principal.

    Feature 6 — fwd_active_ratio:
        Fwd_Act_Data_Pkts / (Total_Fwd_Packets + ε)
        BOTNET:  ~0 (beacon sem dado real).
        DDoS:    ~0 (sem handshake completo).
        Normal:  >0.5 (dados reais trafegando).
        Nota: separa Normal de ataques, mas não BOTNET de DDoS —
        a separação BOTNET/DDoS depende de log_duration e pkt_uniformity.

    Parameters
    ----------
    df : pd.DataFrame — features numéricas com colunas renomeadas

    Returns
    -------
    pd.DataFrame com as seis novas colunas adicionadas (cópia)
    """
    df = df.copy()

    df["asymmetry_pkts"]  = df[_FWD]  / (df[_FWD]  + df[_BWD]  + _EPS)
    df["asymmetry_bytes"] = df[_FWDB] / (df[_FWDB] + df[_BWDB] + _EPS)
    df["pkt_rate"]        = (df[_FWD] + df[_BWD]) / (df[_DUR]  + _EPS)
    df["pkt_uniformity"]  = 1.0 / (df[_STD] + 1.0)
    df["log_duration"]    = np.log1p(np.maximum(df[_DUR], 0.0))

    if _ACT in df.columns:
        df["fwd_active_ratio"] = df[_ACT] / (df[_FWD] + _EPS)
    else:
        # Coluna ausente: preencher com 0 e avisar
        import warnings
        warnings.warn(
            f"[BehavioralFeatureEngineer] Coluna '{_ACT}' ausente. "
            f"fwd_active_ratio preenchida com 0.",
            UserWarning,
            stacklevel=2,
        )
        df["fwd_active_ratio"] = 0.0

    return df


class BehavioralFeatureEngineer:
    """
    Wrapper com interface fit/transform para integração no pipeline.

    Como não há parâmetros aprendidos, fit() é um no-op — existe apenas
    para manter a interface consistente com os demais transformadores.

    Uso:
        eng = BehavioralFeatureEngineer()
        X_train = eng.fit_transform(X_train)   # computa features
        X_test  = eng.transform(X_test)         # mesma transformação
    """

    def __init__(self) -> None:
        self._source_cols_present: list[str] = []

    # ── API pública ────────────────────────────────────────────────────────────

    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """Registra colunas presentes e computa features."""
        self._source_cols_present = [
            c for c in [_FWD, _BWD, _FWDB, _BWDB, _DUR, _STD, _ACT]
            if c in X.columns
        ]
        result = computar_features_comportamentais(X)
        computed = [f for f in BEHAVIORAL_FEATURES if f in result.columns]
        print(f"[BehavioralFeatureEngineer] Features computadas: {computed}")
        return result

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """Aplica a mesma transformação (sem re-fit)."""
        return computar_features_comportamentais(X)

    @property
    def feature_names(self) -> list[str]:
        """Nomes das features comportamentais geradas."""
        return list(BEHAVIORAL_FEATURES)


### Arquivo: `ml/triclass/preprocessing/labeler.py`


In [ ]:
"""
Heurística de labeling triclasse para o InSDN.

Converte os labels originais do InSDN em três classes:
  0 — Benigno      : Label == 'Normal'
  1 — Externo      : Label == 'DDoS' AND is_burst() == True
  2 — Zumbi Interno: Label == 'BOTNET'
                     Label == 'DoS' AND is_burst() == False
 -1 — Descartado   : Probe, BFA, Web-Attack, U2R, DDoS sem burst,
                     DoS com burst — ambíguos ou fora do escopo

SRP: este módulo lida exclusivamente com a construção do vetor y triclasse.

Validação crítica (seção 7.2 do plano):
  BOTNET tem Flow Duration ~31.000 µs >> 500 µs → is_burst() == False (correto)
  DDoS Hping3 tem duration de 1–19 µs           → is_burst() == True  (correto)

Referência: plano_triclasse_insdn_v4.md, Seções 3, 5.1 e 5.2
"""

from __future__ import annotations

import pandas as pd


# ── Funções puras (sem estado — reutilizáveis nos testes) ─────────────────────

def is_burst(df: pd.DataFrame) -> pd.Series:
    """
    Identifica rajada sintética uniforme — padrão de ferramenta de ataque.

    Critério (seção 5.1 do plano):
      Packet Length Std <= 1.0  → pacotes praticamente idênticos
      Flow Duration < 500 µs   → fluxo extremamente curto

    Calibração empírica:
      BOTNET  mediana de Flow Duration ~31.000 µs → is_burst=False (correto)
      DDoS    mediana de Flow Duration 1–19 µs    → is_burst=True  (correto)

    Parameters
    ----------
    df : pd.DataFrame — deve conter 'Packet Length Std' e 'Flow Duration'

    Returns
    -------
    pd.Series[bool] — True onde o fluxo tem padrão de rajada sintética
    """
    return (
        (df["Packet Length Std"] <= BURST_PKT_LEN_STD_MAX) &
        (df["Flow Duration"]     <  BURST_FLOW_DURATION_MAX)
    )


def criar_label_triclasse_insdn(df: pd.DataFrame) -> pd.Series:
    """
    Converte labels originais do InSDN em rótulo triclasse.

    Parameters
    ----------
    df : pd.DataFrame — deve conter colunas 'Label', 'Packet Length Std',
                        'Flow Duration' (já renomeadas via RENAME_MAP)

    Returns
    -------
    pd.Series[int] — valores: 0, 1, 2 ou -1 (descartado)

    Notas
    -----
    - Label == 'DDoS' SEM burst → descartado (-1), não va para Classe 1.
      Razão: DDoS sem burst é ambíguo — pode ser fragmentação ou erro de
      captura. Incluir contaminaria a Classe 1 com padrão não confirmado.
    - Label == 'DoS' COM burst → descartado (-1).
      Razão: DoS com burst é comportamento atípico; proxy DoS aplica-se
      somente a fluxos com duração maior (comportamento de host legítimo
      infectado, não de script de flood).
    """
    label = df["Label"].str.strip()
    burst = is_burst(df)

    y = pd.Series(-1, index=df.index, dtype=int)

    # Classe 0 — Benigno
    y[label == LABEL_BENIGN] = 0

    # Classe 1 — Ataque Externo: DDoS confirmado por burst
    y[(label == "DDoS") & burst] = 1

    # Classe 2 — Zumbi Interno:
    #   BOTNET = ground truth real (âncora semântica)
    #   DoS sem burst = proxy comportamental
    y[label == "BOTNET"]            = 2
    y[(label == "DoS") & ~burst]    = 2

    return y


# ── Classe com interface fit/transform (compatível com o pipeline) ─────────────

class TriclassLabeler:
    """
    Aplica a heurística triclasse e valida o threshold de burst.

    Uso:
        labeler = TriclassLabeler()
        data    = labeler.fit_transform(data)
        # data agora contém coluna 'label_3class'; linhas com -1 são descartadas

    A separação fit/transform é por consistência de interface — a função de
    labeling não aprende parâmetros dos dados; é determinística.
    """

    def __init__(self) -> None:
        self._fitted = False
        self.n_discarded_: int = 0
        self.class_counts_: dict[int, int] = {}
        self.botnet_burst_count_: int = 0

    # ── API pública ────────────────────────────────────────────────────────────

    def fit_transform(self, data: pd.DataFrame) -> pd.DataFrame:
        """
        Aplica labeling triclasse e remove amostras descartadas (-1).

        Valida que nenhum BOTNET é classificado como burst (verificação
        crítica da seção 7.2 do plano).

        Parameters
        ----------
        data : pd.DataFrame — DataFrame completo (pré-split), com colunas
               já renomeadas pelo RENAME_MAP

        Returns
        -------
        pd.DataFrame filtrado (sem linhas -1), com coluna 'label_3class' adicionada
        """
        data = data.copy()
        data["label_3class"] = criar_label_triclasse_insdn(data)

        # Validação crítica: BOTNET nunca deve ser burst
        botnet_mask = data["Label"].str.strip() == "BOTNET"
        self.botnet_burst_count_ = int(
            (botnet_mask & is_burst(data)).sum()
        )
        if self.botnet_burst_count_ > 0:
            import warnings
            warnings.warn(
                f"[TriclassLabeler] ATENÇÃO: {self.botnet_burst_count_} amostras BOTNET "
                f"foram identificadas como burst. O threshold BURST_FLOW_DURATION_MAX "
                f"({BURST_FLOW_DURATION_MAX} µs) pode precisar ser ajustado.",
                UserWarning,
                stacklevel=2,
            )

        n_before = len(data)
        self.n_discarded_ = int((data["label_3class"] == -1).sum())
        data = data[data["label_3class"] != -1].reset_index(drop=True)

        self.class_counts_ = {
            int(cls): int((data["label_3class"] == cls).sum())
            for cls in sorted(data["label_3class"].unique())
        }
        self._fitted = True

        self._print_summary(n_before)
        return data

    def transform(self, data: pd.DataFrame) -> pd.DataFrame:
        """
        Aplica labeling ao conjunto de teste (mesma lógica, sem re-fit).
        """
        if not self._fitted:
            raise RuntimeError(
                "TriclassLabeler: chame fit_transform() antes de transform()."
            )
        data = data.copy()
        data["label_3class"] = criar_label_triclasse_insdn(data)
        return data[data["label_3class"] != -1].reset_index(drop=True)

    # ── Métodos privados ───────────────────────────────────────────────────────

    def _print_summary(self, n_before: int) -> None:
        names = {0: "Benigno", 1: "Externo", 2: "Zumbi Interno"}
        total = sum(self.class_counts_.values())

        print(f"\n[TriclassLabeler] Descartados (fora do escopo): "
              f"{self.n_discarded_:,} / {n_before:,}")
        print(f"[TriclassLabeler] Dataset válido: {total:,}")
        print(f"[TriclassLabeler] Distribuição triclasse:")
        for cls, cnt in self.class_counts_.items():
            pct = 100 * cnt / total
            print(f"  Classe {cls} ({names.get(cls, '?')}): {cnt:,} ({pct:.1f}%)")
        print(f"[TriclassLabeler] Validação BOTNET: "
              f"{self.botnet_burst_count_} amostras BOTNET com burst "
              f"({'OK' if self.botnet_burst_count_ == 0 else 'ATENÇÃO'})")


### Arquivo: `ml/triclass/models/rf_model.py`


In [ ]:
"""
Construção do Random Forest triclasse.

SRP: única responsabilidade é instanciar o modelo com os parâmetros corretos.

Por que RF como modelo principal (seção 6 do plano):
  - Invariante a escala: não afetado pela diferença de magnitude entre
    Flow Duration (µs) e asymmetry_pkts (0–1).
  - class_weight='balanced': compensa desbalanceamento sem dados sintéticos.
  - feature_importances_ nativo: revela quais features distinguem as classes.
  - Robusto a outliers: decisões por threshold em árvores, não por distância.

Referência: plano_triclasse_insdn_v4.md, Seção 6
"""

from __future__ import annotations

from sklearn.ensemble import RandomForestClassifier



def build_baseline_rf(
    n_estimators: int = RF_N_ESTIMATORS,
    class_weight: str = RF_CLASS_WEIGHT,
    random_state: int = RANDOM_STATE,
    **kwargs,
) -> RandomForestClassifier:
    """
    Instancia o Random Forest baseline para classificação triclasse.

    Parameters
    ----------
    n_estimators : int   — número de árvores (default 200, plano seção 8.8)
    class_weight : str   — 'balanced' compensa desbalanceamento (Classe 2)
    random_state : int   — semente de reprodutibilidade
    **kwargs             — parâmetros adicionais repassados ao RF

    Returns
    -------
    RandomForestClassifier com parâmetros do plano, não treinado.
    """
    return RandomForestClassifier(
        n_estimators=n_estimators,
        class_weight=class_weight,
        random_state=random_state,
        n_jobs=-1,
        **kwargs,
    )


### Arquivo: `ml/triclass/training/trainer.py`


In [ ]:
"""
Treinamento e validação cruzada dos modelos RF e MLP triclasse.

SRP: gerencia o ciclo treino + CV estratificado.

Regra do curso:
  - Validação cruzada SOMENTE no treino (nunca no test_set).
  - MLP obrigatoriamente dentro de Pipeline(StandardScaler → MLP) para
    evitar leakage do scaler no CV (Aula 5).

Referência: plano_triclasse_insdn_v4.md, Seção 8.8
"""

from __future__ import annotations

import time

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler



def build_mlp_pipeline(random_state: int = RANDOM_STATE) -> Pipeline:
    """
    Constrói Pipeline(StandardScaler → MLP).

    O scaler dentro do Pipeline garante que o fit do scaler ocorra
    somente nos folds de treino durante o CV — zero leakage.
    """
    return Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPClassifier(
            hidden_layer_sizes=MLP_HIDDEN_LAYERS,
            activation=MLP_ACTIVATION,
            solver=MLP_SOLVER,
            max_iter=MLP_MAX_ITER,
            early_stopping=MLP_EARLY_STOP,
            n_iter_no_change=MLP_N_ITER_NO_CHG,
            random_state=random_state,
        )),
    ])


class TriclassTrainer:
    """
    Treina RF e MLP triclasse com validação cruzada estratificada.

    Uso:
        trainer = TriclassTrainer()
        rf, mlp = trainer.train(X_train_bal, y_train_bal)
        cv_rf   = trainer.cross_validate_rf(X_train_bal, y_train_bal)
        cv_mlp  = trainer.cross_validate_mlp(X_train_bal, y_train_bal)
    """

    def __init__(
        self,
        random_state: int = RANDOM_STATE,
        cv_n_splits: int = CV_N_SPLITS,
        cv_scoring: str = CV_SCORING,
        save_plots: bool = True,
    ) -> None:
        self._random_state = random_state
        self._cv_n_splits  = cv_n_splits
        self._cv_scoring   = cv_scoring
        self._save_plots   = save_plots

    # ── API pública ────────────────────────────────────────────────────────────

    def train_rf(
        self,
        X_train: np.ndarray | pd.DataFrame,
        y_train: np.ndarray | pd.Series,
        rf=None,
    ):
        """Treina Random Forest no treino balanceado."""
        if rf is None:
            rf = build_baseline_rf(random_state=self._random_state)

        print("\n[TriclassTrainer] Treinando Random Forest baseline...")
        print(f"  n_estimators : {rf.n_estimators}")
        print(f"  class_weight : {rf.class_weight}")
        print(f"  Shape treino : {np.array(X_train).shape}")

        t0 = time.monotonic()
        rf.fit(X_train, y_train)
        elapsed = time.monotonic() - t0
        print(f"[TriclassTrainer] RF treinado em {elapsed:.1f}s")
        return rf

    def train_mlp(
        self,
        X_train: np.ndarray | pd.DataFrame,
        y_train: np.ndarray | pd.Series,
        mlp_pipe: Pipeline | None = None,
    ) -> Pipeline:
        """Treina Pipeline(StandardScaler → MLP) no treino balanceado."""
        if mlp_pipe is None:
            mlp_pipe = build_mlp_pipeline(random_state=self._random_state)

        print("\n[TriclassTrainer] Treinando MLP (Pipeline)...")
        mlp = mlp_pipe.named_steps["mlp"]
        print(f"  Arquitetura  : {mlp.hidden_layer_sizes}")
        print(f"  Shape treino : {np.array(X_train).shape}")

        t0 = time.monotonic()
        mlp_pipe.fit(X_train, y_train)
        elapsed = time.monotonic() - t0

        mlp_fitted = mlp_pipe.named_steps["mlp"]
        print(f"[TriclassTrainer] MLP treinado em {elapsed:.1f}s")
        print(f"  Épocas : {mlp_fitted.n_iter_}")
        print(f"  Loss   : {mlp_fitted.loss_:.6f}")

        if self._save_plots:
            self._plot_mlp_loss(mlp_fitted)

        return mlp_pipe

    def cross_validate_rf(
        self,
        X_train: np.ndarray | pd.DataFrame,
        y_train: np.ndarray | pd.Series,
    ) -> dict[str, tuple[float, float]]:
        """
        Validação cruzada do RF no conjunto de TREINO.

        Retorna dict: métrica → (mean, std)
        """
        return self._cross_validate(
            build_baseline_rf(random_state=self._random_state),
            X_train, y_train,
            label="Random Forest",
        )

    def cross_validate_mlp(
        self,
        X_train: np.ndarray | pd.DataFrame,
        y_train: np.ndarray | pd.Series,
    ) -> dict[str, tuple[float, float]]:
        """
        Validação cruzada do MLP no conjunto de TREINO.

        Pipeline(StandardScaler → MLP) garante que o scaler seja fitado
        somente nos folds de treino — evita leakage no CV.
        """
        return self._cross_validate(
            build_mlp_pipeline(random_state=self._random_state),
            X_train, y_train,
            label="MLP Pipeline",
        )

    # ── Métodos privados ───────────────────────────────────────────────────────

    def _cross_validate(
        self,
        estimator,
        X_train,
        y_train,
        label: str,
    ) -> dict[str, tuple[float, float]]:
        cv = StratifiedKFold(
            n_splits=self._cv_n_splits,
            shuffle=True,
            random_state=self._random_state,
        )

        # F1 macro é a métrica primária para triclasse desbalanceada
        scoring_metrics = ["f1_macro", "accuracy"]
        results: dict[str, tuple[float, float]] = {}

        print(f"\n[TriclassTrainer] CV {self._cv_n_splits}-fold — {label} (treino):")
        print("-" * 55)

        for metric in scoring_metrics:
            scores = cross_val_score(
                estimator, X_train, y_train,
                cv=cv, scoring=metric, n_jobs=-1,
            )
            mean, std = float(scores.mean()), float(scores.std())
            results[metric] = (mean, std)
            print(f"  {metric:<20}: {mean:.4f} ± {std:.4f}")

        print("-" * 55)
        return results

    def _plot_mlp_loss(self, mlp: MLPClassifier) -> None:
        try:
            import matplotlib.pyplot as plt
            OUTPUTS_TRICLASS.mkdir(parents=True, exist_ok=True)

            fig, ax = plt.subplots(figsize=(10, 4))
            ax.plot(mlp.loss_curve_, label="Loss de Treino", color="steelblue")
            if hasattr(mlp, "validation_scores_") and mlp.validation_scores_:
                ax.plot(
                    mlp.validation_scores_,
                    label="Score Validação Interna",
                    color="orange", linestyle="--",
                )
            ax.set_xlabel("Épocas")
            ax.set_ylabel("Loss / Score")
            ax.set_title("Curva de Convergência — MLP Triclasse (InSDN)")
            ax.legend()
            plt.tight_layout()
            path = OUTPUTS_TRICLASS / "mlp_loss_curve.png"
            plt.savefig(path, dpi=150, bbox_inches="tight")
            plt.close()
            print(f"[TriclassTrainer] Curva MLP salva → {path.name}")
        except Exception as e:
            print(f"[TriclassTrainer] Não foi possível salvar curva MLP: {e}")


### Arquivo: `ml/triclass/training/tuner.py`


In [ ]:
"""
Hyperparameter tuning do Random Forest triclasse.

SRP: encapsula exclusivamente o RandomizedSearchCV no conjunto de treino.

Regra absoluta (Aula 11 — boas práticas):
  O test_set NUNCA entra no tuning.
  O tuner usa apenas X_train_bal + y_train_bal via CV interno.

Referência: plano_triclasse_insdn_v4.md, Seção 8.9
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier



class TriclassTuner:
    """
    RandomizedSearchCV para o Random Forest triclasse.

    Uso:
        tuner    = TriclassTuner()
        rf_best  = tuner.fit(X_train_bal, y_train_bal)
        print(tuner.best_params_)
        print(tuner.best_score_)
    """

    def __init__(
        self,
        param_dist: dict | None = None,
        n_iter: int = RF_TUNING_N_ITER,
        cv_splits: int = 5,
        scoring: str = "f1_macro",
        random_state: int = RANDOM_STATE,
    ) -> None:
        self._param_dist   = param_dist or RF_TUNING_PARAM_DIST
        self._n_iter       = n_iter
        self._cv_splits    = cv_splits
        self._scoring      = scoring
        self._random_state = random_state

        self.best_params_: dict = {}
        self.best_score_: float = 0.0
        self._search: RandomizedSearchCV | None = None

    # ── API pública ────────────────────────────────────────────────────────────

    def fit(
        self,
        X_train: np.ndarray | pd.DataFrame,
        y_train: np.ndarray | pd.Series,
    ) -> RandomForestClassifier:
        """
        Executa busca aleatória de hiperparâmetros no conjunto de TREINO.

        Parameters
        ----------
        X_train : features balanceadas (pós-SMOTE)
        y_train : labels balanceados (pós-SMOTE)

        Returns
        -------
        RandomForestClassifier com os melhores parâmetros encontrados,
        retreinado sobre todo X_train.
        """
        cv = StratifiedKFold(
            n_splits=self._cv_splits,
            shuffle=True,
            random_state=self._random_state,
        )

        base_rf = RandomForestClassifier(
            class_weight=RF_CLASS_WEIGHT,
            random_state=self._random_state,
            n_jobs=-1,
        )

        self._search = RandomizedSearchCV(
            estimator=base_rf,
            param_distributions=self._param_dist,
            n_iter=self._n_iter,
            cv=cv,
            scoring=self._scoring,
            random_state=self._random_state,
            n_jobs=-1,
            verbose=1,
        )

        print(f"\n[TriclassTuner] RandomizedSearchCV "
              f"({self._n_iter} iter × {self._cv_splits}-fold)...")
        self._search.fit(X_train, y_train)

        self.best_params_ = self._search.best_params_
        self.best_score_  = float(self._search.best_score_)

        print(f"[TriclassTuner] Melhores parâmetros : {self.best_params_}")
        print(f"[TriclassTuner] Melhor F1 Macro CV  : {self.best_score_:.4f}")

        return self._search.best_estimator_

    @property
    def cv_results(self) -> dict | None:
        """Resultados completos do CV para análise posterior."""
        return self._search.cv_results_ if self._search else None


### Arquivo: `ml/triclass/evaluation/evaluator.py`


In [ ]:
"""
Avaliação triclasse com métricas adequadas para dados desbalanceados.

Métricas escolhidas (seção 6 do plano — Aula 7):
  - F1 Macro      : média não-ponderada do F1 por classe — penaliza classes
                    com baixo desempenho independente do volume.
  - MCC           : Matthews Correlation Coefficient — melhor métrica única
                    para desbalanceamento multiclasse.
  - Geometric Mean: raiz do produto dos recalls — equilibra sensibilidade
                    de todas as classes.
  - Recall Classe 2: o mais crítico em produção — falso negativo em Zumbi
                    Interno = host comprometido não isolado.

NÃO usar: acurácia sozinha, F1-weighted (mascara classes minoritárias).

SRP: calcula métricas e gera plots — nunca toca no modelo ou nos dados de treino.

Referência: plano_triclasse_insdn_v4.md, Seção 8.10
"""

from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from imblearn.metrics import geometric_mean_score
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    recall_score,
)



@dataclass
class TriclassEvaluationResult:
    """
    Resultado imutável de uma avaliação triclasse.

    Armazena métricas globais e por classe para comparação entre modelos.
    """

    label: str
    f1_macro: float
    mcc: float
    gm: float

    # Métricas por classe (índice = classe)
    f1_per_class: list[float]
    precision_per_class: list[float]
    recall_per_class: list[float]
    support_per_class: list[int]

    # Texto do classification_report (sklearn)
    report: str = field(default="", repr=False)

    @property
    def recall_class2(self) -> float:
        """Recall da Classe 2 (Zumbi Interno) — métrica mais crítica em produção."""
        return self.recall_per_class[2] if len(self.recall_per_class) > 2 else 0.0

    @property
    def f1_class2(self) -> float:
        return self.f1_per_class[2] if len(self.f1_per_class) > 2 else 0.0

    def print_summary(self) -> None:
        width = 62
        print("\n" + "=" * width)
        print(f"{f'RESULTADOS — {self.label}':^{width}}")
        print("=" * width)
        print(f"  F1 Macro      : {self.f1_macro:.4f}")
        print(f"  MCC           : {self.mcc:.4f}")
        print(f"  Geometric Mean: {self.gm:.4f}")
        print()
        print(f"  {'Classe':<22} {'F1':>7} {'Precision':>10} {'Recall':>8} {'Support':>9}")
        print(f"  {'-'*58}")
        for i, name in CLASS_NAMES.items():
            if i < len(self.f1_per_class):
                print(
                    f"  {i} {name:<20} "
                    f"{self.f1_per_class[i]:>7.4f} "
                    f"{self.precision_per_class[i]:>10.4f} "
                    f"{self.recall_per_class[i]:>8.4f} "
                    f"{self.support_per_class[i]:>9,}"
                )
        print()
        print(f"  ⚠ Recall Classe 2 (Zumbi): {self.recall_class2:.4f}"
              f"  {'OK' if self.recall_class2 >= 0.60 else 'BAIXO — revisar'}")
        print("=" * width)


class TriclassEvaluator:
    """
    Avalia modelos triclasse no test_set e gera relatórios/plots.

    Regra absoluta: test_set usado UMA ÚNICA VEZ, ao final de todo o pipeline.

    Uso:
        evaluator = TriclassEvaluator()
        result_rf  = evaluator.evaluate(rf_best,  X_test_vt, y_test, "RF Otimizado")
        result_mlp = evaluator.evaluate(mlp_pipe, X_test_vt, y_test, "MLP")
        evaluator.compare(result_rf, result_mlp)
    """

    def __init__(self, save_plots: bool = True) -> None:
        self._save_plots = save_plots
        OUTPUTS_TRICLASS.mkdir(parents=True, exist_ok=True)

    # ── API pública ────────────────────────────────────────────────────────────

    def evaluate(
        self,
        model,
        X_test: np.ndarray,
        y_test: np.ndarray,
        label: str = "Modelo",
    ) -> TriclassEvaluationResult:
        """
        Avalia o modelo no test_set (usado UMA ÚNICA VEZ).

        Parameters
        ----------
        model  : estimador sklearn treinado (RF ou Pipeline com MLP)
        X_test : features do teste (pós-VT, sem SMOTE)
        y_test : alvo real do teste
        label  : nome do modelo para exibição e nomes de arquivos

        Returns
        -------
        TriclassEvaluationResult com todas as métricas.
        """
        y_pred = model.predict(X_test)
        classes = sorted(np.unique(np.concatenate([y_test, y_pred])))

        cm = confusion_matrix(y_test, y_pred, labels=classes)

        f1_per   = f1_score(y_test, y_pred, average=None, zero_division=0, labels=classes)
        prec_per = __import__("sklearn.metrics", fromlist=["precision_score"]).precision_score(
            y_test, y_pred, average=None, zero_division=0, labels=classes
        )
        rec_per  = recall_score(y_test, y_pred, average=None, zero_division=0, labels=classes)
        sup_per  = [int((np.array(y_test) == c).sum()) for c in classes]

        result = TriclassEvaluationResult(
            label=label,
            f1_macro=float(f1_score(y_test, y_pred, average="macro", zero_division=0)),
            mcc=float(matthews_corrcoef(y_test, y_pred)),
            gm=float(geometric_mean_score(y_test, y_pred, average="macro")),
            f1_per_class=f1_per.tolist(),
            precision_per_class=prec_per.tolist(),
            recall_per_class=rec_per.tolist(),
            support_per_class=sup_per,
            report=classification_report(
                y_test, y_pred,
                target_names=[CLASS_NAMES.get(c, str(c)) for c in classes],
                zero_division=0,
            ),
        )

        result.print_summary()
        print("\nRelatório completo:")
        print(result.report)

        if self._save_plots:
            slug = label.lower().replace(" ", "_").replace("(", "").replace(")", "")
            self._plot_confusion_matrix(cm, label, slug)

        return result

    def compare(self, *results: TriclassEvaluationResult) -> None:
        """
        Tabela comparativa entre N modelos avaliados.
        """
        width = 72
        print("\n" + "=" * width)
        print(f"{'COMPARAÇÃO DE MODELOS TRICLASSE':^{width}}")
        print("=" * width)

        header = f"{'Métrica':<28}"
        for r in results:
            header += f" {r.label[:12]:>12}"
        print(header)
        print("-" * width)

        def row(name, values):
            line = f"{name:<28}"
            for v in values:
                line += f" {v:>12.4f}"
            print(line)

        row("F1 Macro",      [r.f1_macro for r in results])
        row("MCC",           [r.mcc      for r in results])
        row("Geometric Mean",[r.gm       for r in results])
        row("Recall Cl.2 (Zumbi)", [r.recall_class2 for r in results])
        row("F1 Cl.2 (Zumbi)",     [r.f1_class2     for r in results])
        print("=" * width)

    # ── Métodos privados ───────────────────────────────────────────────────────

    def _plot_confusion_matrix(
        self,
        cm: np.ndarray,
        label: str,
        slug: str,
    ) -> None:
        try:
            display_labels = [CLASS_NAMES.get(i, str(i)) for i in range(cm.shape[0])]
            fig, ax = plt.subplots(figsize=(8, 6))
            ConfusionMatrixDisplay(
                confusion_matrix=cm,
                display_labels=display_labels,
            ).plot(values_format=".0f", ax=ax, cmap="Blues", colorbar=False)
            ax.set_title(f"Matriz de Confusão — {label} (InSDN Triclasse)")
            plt.tight_layout()
            path = OUTPUTS_TRICLASS / f"confusion_matrix_{slug}.png"
            plt.savefig(path, dpi=150, bbox_inches="tight")
            plt.close()
            print(f"[TriclassEvaluator] Matriz salva → {path.name}")
        except Exception as e:
            print(f"[TriclassEvaluator] Erro ao salvar matriz: {e}")


### Arquivo: `ml/triclass/evaluation/semantic_validator.py`


In [ ]:
"""
Validação semântica do BOTNET no test_set.

A validação semântica (seção 8.11 do plano) é a evidência mais robusta
do que métricas agregadas: verifica se o modelo reconhece especificamente
o padrão de beacon/C2 dos 164 fluxos BOTNET reais.

Se recall ≥ 80% → as features log_duration e pkt_uniformity funcionam
como diferenciadores BOTNET vs DDoS, mesmo com volume reduzido.

SRP: único propósito é calcular e reportar o recall semântico do BOTNET.

Referência: plano_triclasse_insdn_v4.md, Seção 8.11
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd



@dataclass
class BotnetValidationResult:
    """Resultado da validação semântica do BOTNET."""
    n_botnet_test: int
    n_classified_as_class2: int
    n_classified_as_class1: int
    n_classified_as_class0: int
    recall: float
    passed: bool

    def print_report(self) -> None:
        print("\n" + "=" * 55)
        print("  Validação Semântica — BOTNET (ground truth)")
        print("=" * 55)
        if self.n_botnet_test == 0:
            print("  ⚠ Nenhuma amostra BOTNET no test_set.")
            print("  Aumente test_size ou mude random_state.")
            return
        print(f"  Amostras BOTNET no teste : {self.n_botnet_test}")
        print(f"  → Classe 2 (correto)     : {self.n_classified_as_class2} "
              f"({100*self.recall:.1f}%)")
        print(f"  → Classe 1 (confundido)  : {self.n_classified_as_class1}")
        print(f"  → Classe 0 (confundido)  : {self.n_classified_as_class0}")
        print(f"\n  Recall semântico BOTNET  : {self.recall:.4f}")
        print(f"  Threshold mínimo         : {BOTNET_MIN_RECALL:.2f}")
        if self.passed:
            print("\n  ✓ Modelo reconhece padrão beacon/C2 corretamente.")
            print("    log_duration e pkt_uniformity funcionam como diferenciadores.")
        else:
            print("\n  ⚠ Modelo confunde BOTNET com outra classe.")
            print("    Verificar: log_duration diferencia 31ms de 1-19µs?")
            print("    Possível causa: SMOTE gerou exemplos que se misturaram com DDoS.")
        print("=" * 55)


class BotnetSemanticValidator:
    """
    Valida se o modelo classifica corretamente os fluxos BOTNET reais.

    Requer acesso ao DataFrame original para identificar quais amostras
    do test_set têm label original == 'BOTNET'.

    Uso:
        validator = BotnetSemanticValidator(data_original, test_indices)
        result = validator.validate(model, X_test_vt)
        result.print_report()
    """

    def __init__(
        self,
        data_original: pd.DataFrame,
        test_indices: np.ndarray | pd.Index,
    ) -> None:
        """
        Parameters
        ----------
        data_original : DataFrame completo com coluna 'Label' original preservada
        test_indices  : índices das amostras que foram para o test_set
        """
        self._data   = data_original
        self._test_ix = np.array(test_indices)

    # ── API pública ────────────────────────────────────────────────────────────

    def validate(
        self,
        model,
        X_test_vt: np.ndarray | pd.DataFrame,
        min_recall: float = BOTNET_MIN_RECALL,
    ) -> BotnetValidationResult:
        """
        Avalia o recall do modelo especificamente nos fluxos BOTNET do teste.

        Parameters
        ----------
        model      : estimador sklearn treinado
        X_test_vt  : features do teste após VarianceThreshold
        min_recall : recall mínimo para considerar validação aprovada

        Returns
        -------
        BotnetValidationResult
        """
        # Identifica amostras BOTNET no subconjunto de teste
        label_test = self._data.loc[self._test_ix, "Label"].str.strip()
        botnet_mask = (label_test == "BOTNET").values

        n_botnet = int(botnet_mask.sum())

        if n_botnet == 0:
            return BotnetValidationResult(
                n_botnet_test=0,
                n_classified_as_class2=0,
                n_classified_as_class1=0,
                n_classified_as_class0=0,
                recall=0.0,
                passed=False,
            )

        X_botnet = (
            X_test_vt[botnet_mask]
            if isinstance(X_test_vt, np.ndarray)
            else X_test_vt.iloc[botnet_mask] if hasattr(X_test_vt, "iloc")
            else X_test_vt[botnet_mask]
        )

        y_pred_botnet = model.predict(X_botnet)

        n_cls2 = int((y_pred_botnet == 2).sum())
        n_cls1 = int((y_pred_botnet == 1).sum())
        n_cls0 = int((y_pred_botnet == 0).sum())
        recall = n_cls2 / n_botnet

        result = BotnetValidationResult(
            n_botnet_test=n_botnet,
            n_classified_as_class2=n_cls2,
            n_classified_as_class1=n_cls1,
            n_classified_as_class0=n_cls0,
            recall=recall,
            passed=recall >= min_recall,
        )
        result.print_report()
        return result


### Arquivo: `ml/triclass/inference/predictor.py`


In [ ]:
"""
Preditor triclasse integrado — Estágio 2 do sistema de detecção.

Arquitetura dois estágios (seção 9.1 do plano):
  Estágio 1 — MLP binário existente:
    → Benigno (0): liberar. Fim.
    → Ataque (1):  passar para Estágio 2.

  Estágio 2 — RF triclasse (novo):
    → Classe 1 (Externo):  POST /manage/ip   → block global
    → Classe 2 (Interno):  POST /mitigation/isolate/{ip} → isolamento cirúrgico

  Fallback determinístico (Opção C — produção):
    → is_known_host == True + predição == 1 → forçar Classe 2
      Razão: IP registrado no SDN é evidência mais confiável que o modelo.

SRP: recebe features brutas → retorna dict com classe, label e ação.

Referência: plano_triclasse_insdn_v4.md, Seção 9.2
"""

from __future__ import annotations

from pathlib import Path

import joblib
import numpy as np
import pandas as pd


# Ações de resposta do SDN por classe
_ACTIONS: dict[int, str] = {
    0: "none",
    1: "block_global",
    2: "isolate_surgical",
}


class DDoSPredictorV2:
    """
    Preditor em dois estágios para produção.

    Carrega artefatos persistidos pelo pipeline e fornece interface
    simplificada para o controlador SDN.

    Uso:
        predictor = DDoSPredictorV2()
        result = predictor.predict(X_raw_df, is_known_host=False)
        # result = {'class': 1, 'label': 'Externo', 'action': 'block_global',
        #           'confidence': 0.97}
    """

    def __init__(
        self,
        models_binary_dir: str | Path | None = None,
        models_triclass_dir: str | Path = MODELS_TRICLASS_DIR,
    ) -> None:
        tc_dir = Path(models_triclass_dir)

        self._rf          = joblib.load(tc_dir / "rf_triclass.joblib")
        self._imputer     = joblib.load(tc_dir / "imputer.joblib")
        self._vt          = joblib.load(tc_dir / "variance_filter.joblib")
        self._feat_names  = joblib.load(tc_dir / "selected_features.joblib")

        # MLP binário (Estágio 1) — opcional; se ausente, pula o Estágio 1
        self._mlp_binary = None
        if models_binary_dir is not None:
            bin_dir = Path(models_binary_dir)
            mlp_path = bin_dir / "mlp_model.joblib"
            if mlp_path.exists():
                self._mlp_binary = joblib.load(mlp_path)

    # ── API pública ────────────────────────────────────────────────────────────

    def predict(
        self,
        X_raw: pd.DataFrame,
        is_known_host: bool = False,
    ) -> dict:
        """
        Classifica um fluxo de rede nas três classes.

        Parameters
        ----------
        X_raw         : DataFrame com features brutas do CICFlowMeter/InSDN
        is_known_host : True se o IP de origem está registrado no state.ip_to_mac
                        do controlador SDN (evidência de host interno)

        Returns
        -------
        dict com keys: class (int), label (str), action (str), confidence (float),
                       e opcionalmente reason (str) para o fallback determinístico
        """
        # ── Estágio 1: filtro binário (se disponível) ─────────────────────────
        if self._mlp_binary is not None:
            binary_pred = self._mlp_binary.predict(X_raw)[0]
            if binary_pred == 0:
                return {
                    "class": 0,
                    "label": CLASS_NAMES[0],
                    "action": _ACTIONS[0],
                    "confidence": 1.0,
                }

        # ── Pré-processamento (mesmas transformações do treino) ────────────────
        X = X_raw.copy()
        X = X.rename(columns={k: v for k, v in RENAME_MAP.items() if k in X.columns})
        X = computar_features_comportamentais(X)

        X_num = X.select_dtypes(include=np.number)
        X_imp = pd.DataFrame(
            self._imputer.transform(X_num.replace([np.inf, -np.inf], np.nan)),
            columns=X_num.columns,
        )
        # Alinhar com features selecionadas no treino
        available = [f for f in self._feat_names if f in X_imp.columns]
        X_vt = X_imp[available].values

        # ── Estágio 2: predição triclasse ─────────────────────────────────────
        classe     = int(self._rf.predict(X_vt)[0])
        confianca  = float(self._rf.predict_proba(X_vt)[0][classe])

        # ── Fallback determinístico (Opção C) ─────────────────────────────────
        # IP registrado no SDN é evidência mais forte que o modelo ML
        if is_known_host and classe == 1:
            return {
                "class": 2,
                "label": CLASS_NAMES[2],
                "action": _ACTIONS[2],
                "confidence": 0.95,
                "reason": "IP registrado no SDN — reclassificado como Interno",
            }

        return {
            "class": classe,
            "label": CLASS_NAMES.get(classe, f"Classe {classe}"),
            "action": _ACTIONS.get(classe, "unknown"),
            "confidence": confianca,
        }

    def predict_batch(
        self,
        X_raw: pd.DataFrame,
        is_known_hosts: list[bool] | None = None,
    ) -> list[dict]:
        """
        Classifica um batch de fluxos de rede.

        Parameters
        ----------
        X_raw          : DataFrame com múltiplas linhas (um fluxo por linha)
        is_known_hosts : lista de bools, um por linha; None = todos False

        Returns
        -------
        Lista de dicts (um resultado por fluxo)
        """
        if is_known_hosts is None:
            is_known_hosts = [False] * len(X_raw)

        return [
            self.predict(X_raw.iloc[[i]], is_known_host=is_known_hosts[i])
            for i in range(len(X_raw))
        ]


### Arquivo: `ml/triclass/pipeline.py`


In [ ]:
"""
Pipeline principal — Detecção Triclasse de DDoS em SDN (InSDN v4).

Implementa rigorosamente as 14 etapas do plano e do guia de boas práticas:

  1.  Configurações e reprodutibilidade
  2.  Carregamento dos três CSVs do InSDN
  3.  EDA (sem modificar dados)
  4.  Labeling triclasse (heurística is_burst)
  5.  ⚠️  SPLIT treino/teste (stratify=y) — ANTES de qualquer transformação
  6.  Limpeza: duplicatas, Inf→NaN, imputação (fit somente no treino)
  7.  Features comportamentais (puras — sem leakage)
  8.  VarianceThreshold (fit somente no treino)
  9.  SMOTE conservador (somente no treino)
  10. CV 10-fold no treino — RF baseline e MLP Pipeline
  11. Hyperparameter Tuning RF (RandomizedSearchCV no treino)
  12. Avaliação final no test_set (UMA ÚNICA VEZ)
  13. Validação semântica BOTNET
  14. Importância de features + salvamento de artefatos

Referência: plano_triclasse_insdn_v4.md + guia_boas_praticas_ml.md
"""

from __future__ import annotations

import matplotlib
matplotlib.use("Agg")  # backend não-interativo — obrigatório com n_jobs=-1

import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler


warnings.filterwarnings("ignore")
np.random.seed(RANDOM_STATE)


def run_triclass_pipeline(
    run_tuning: bool = True,
    run_eda: bool = True,
    save_plots: bool = True,
    run_id: str | None = None,
) -> dict:
    """
    Executa o pipeline triclasse completo.

    Parameters
    ----------
    run_tuning  : bool — executa RandomizedSearchCV (mais lento)
    run_eda     : bool — exibe análise exploratória textual
    save_plots  : bool — salva gráficos em outputs_triclass/
    run_id      : str  — identificador para log de métricas

    Returns
    -------
    dict com resultados: 'rf_baseline', 'rf_best', 'mlp', 'botnet_validation'
    """
    OUTPUTS_TRICLASS.mkdir(parents=True, exist_ok=True)
    MODELS_TRICLASS_DIR.mkdir(parents=True, exist_ok=True)

    _banner("Pipeline Triclasse — Detecção de DDoS em SDN (InSDN v4)")

    # ── Etapa 1: Configurações ─────────────────────────────────────────────────
    _step(1, "Configurações")
    print(f"  RANDOM_STATE     : {RANDOM_STATE}")
    print(f"  TEST_SIZE        : {TEST_SIZE} (70/30)")
    print(f"  VARIANCE_THRESH  : {VARIANCE_THRESHOLD}")
    print(f"  SMOTE fator cls2 : {SMOTE_MAX_FACTOR_CLS2}x")

    # ── Etapa 2: Carregamento ──────────────────────────────────────────────────
    _step(2, "Carregando InSDN (3 arquivos CSV)")
    loader = InSDNTriclassLoader()
    data_raw = loader.load()

    # ── Etapa 3: EDA ──────────────────────────────────────────────────────────
    if run_eda:
        _step(3, "EDA (sem modificar dados)")
        loader.describe(data_raw)
    else:
        _step(3, "EDA ignorada (run_eda=False)")

    # ── Etapa 4: Labeling triclasse ────────────────────────────────────────────
    _step(4, "Labeling triclasse (heurística is_burst)")
    labeler = TriclassLabeler()
    data = labeler.fit_transform(data_raw)

    X_all = data.drop(columns=["Label", "label_3class", "_source"], errors="ignore")
    X_all = X_all.select_dtypes(include=np.number)
    y_all = data["label_3class"]

    # Preservar DataFrame original com labels para validação semântica
    # (precisamos dos índices originais após o split)
    data_for_validation = data.copy()

    # ── Etapa 5: SPLIT (ANTES de qualquer transformação) ──────────────────────
    _step(5, "Split estratificado 70/30 (ANTES de qualquer transformação)")
    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_all,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True,
        stratify=y_all,
    )
    # Preservar índices do teste para validação semântica
    test_original_indices = X_test.index.values

    X_train = X_train.reset_index(drop=True)
    X_test  = X_test.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)
    y_test  = y_test.reset_index(drop=True)

    print(f"  Treino : {X_train.shape[0]:,} | Teste: {X_test.shape[0]:,}")
    _print_dist("Distribuição treino", y_train)
    _print_dist("Distribuição teste ", y_test)

    # ── Etapa 6: Limpeza (somente no treino) ──────────────────────────────────
    _step(6, "Limpeza: duplicatas, Inf→NaN, imputação (fit somente no treino)")

    # 6a. Duplicatas no treino (junto com y para consistência)
    df_tmp = X_train.copy()
    df_tmp["__y__"] = y_train.values
    before = len(df_tmp)
    df_tmp = df_tmp.drop_duplicates(keep="first")
    after  = len(df_tmp)
    print(f"  Duplicatas removidas do treino: {before - after:,}")
    X_train = df_tmp.drop(columns=["__y__"])
    y_train = pd.Series(df_tmp["__y__"].values, name="label_3class")

    # 6b. Inf → NaN
    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    X_test  = X_test.replace([np.inf, -np.inf], np.nan)

    # 6c. Imputação com mediana (fit somente no treino)
    imputer = SimpleImputer(strategy=IMPUTER_STRATEGY)
    X_train = pd.DataFrame(
        imputer.fit_transform(X_train), columns=X_train.columns
    )
    X_test  = pd.DataFrame(
        imputer.transform(X_test), columns=X_test.columns
    )
    print(f"  NaN pós-imputação treino : {X_train.isnull().sum().sum()}")
    print(f"  NaN pós-imputação teste  : {X_test.isnull().sum().sum()}")

    # ── Etapa 7: Features comportamentais ─────────────────────────────────────
    _step(7, "Features comportamentais (puras — sem leakage)")
    eng = BehavioralFeatureEngineer()
    X_train = eng.fit_transform(X_train)
    X_test  = eng.transform(X_test)

    computed_feats = [f for f in BEHAVIORAL_FEATURES if f in X_train.columns]

    # Guarda: feature engineering não deve introduzir NaN
    nan_train = X_train.isnull().sum().sum()
    nan_test  = X_test.isnull().sum().sum()
    if nan_train or nan_test:
        bad_cols = X_train.columns[X_train.isnull().any()].tolist()
        raise ValueError(
            f"NaN introduzido pela feature engineering — colunas: {bad_cols}"
        )

    # ── Etapa 8: VarianceThreshold (fit somente no treino) ────────────────────
    _step(8, "VarianceThreshold (fit somente no treino)")
    vt = VarianceThreshold(threshold=VARIANCE_THRESHOLD)
    X_train_vt = pd.DataFrame(
        vt.fit_transform(X_train),
        columns=X_train.columns[vt.get_support()],
    )
    X_test_vt = pd.DataFrame(
        vt.transform(X_test),
        columns=X_train.columns[vt.get_support()],
    )
    removed = (~vt.get_support()).sum()
    print(f"  Features removidas (var≤{VARIANCE_THRESHOLD}): {removed}")
    print(f"  Features restantes: {X_train_vt.shape[1]}")
    selected_features = X_train_vt.columns.tolist()

    # ── Etapa 9: SMOTE conservador (somente no treino) ────────────────────────
    _step(9, "SMOTE conservador (somente no treino)")
    n0 = int((y_train == 0).sum())
    n1 = int((y_train == 1).sum())
    n2 = int((y_train == 2).sum())
    print(f"  Antes: Cl0={n0:,}  Cl1={n1:,}  Cl2={n2:,}")

    target_cls2 = min(n2 * SMOTE_MAX_FACTOR_CLS2, n0)
    target_cls2 = max(target_cls2, n2 + 1)  # garante ao menos 1 amostra extra

    smote = SMOTE(
        sampling_strategy={2: target_cls2},
        random_state=RANDOM_STATE,
        k_neighbors=min(5, n2 - 1),  # evita erro quando n2 é muito pequeno
    )
    X_res, y_res = smote.fit_resample(X_train_vt, y_train)

    # Undersample classe 1 se muito maior que classe 0
    n1_res = int((y_res == 1).sum())
    n0_res = int((y_res == 0).sum())
    if n1_res > n0_res * 2:
        target_cls1 = int(n0_res * SMOTE_UNDERSAMPLE_RATIO_CLS1)
        under = RandomUnderSampler(
            sampling_strategy={1: target_cls1},
            random_state=RANDOM_STATE,
        )
        X_train_bal, y_train_bal = under.fit_resample(X_res, y_res)
    else:
        X_train_bal, y_train_bal = X_res, y_res

    n0b = int((y_train_bal == 0).sum())
    n1b = int((y_train_bal == 1).sum())
    n2b = int((y_train_bal == 2).sum())
    print(f"  Depois: Cl0={n0b:,}  Cl1={n1b:,}  Cl2={n2b:,}")

    # ── Etapa 10: CV 10-fold no treino ────────────────────────────────────────
    _step(10, "Validação cruzada 10-fold no treino")
    trainer = TriclassTrainer(save_plots=save_plots)
    cv_rf  = trainer.cross_validate_rf(X_train_bal, y_train_bal)
    cv_mlp = trainer.cross_validate_mlp(X_train_bal, y_train_bal)

    # ── Treinar modelos completos no treino balanceado ─────────────────────────
    rf_baseline = trainer.train_rf(X_train_bal, y_train_bal)
    mlp_pipe    = trainer.train_mlp(X_train_bal, y_train_bal)

    # ── Etapa 11: Hyperparameter Tuning (somente no treino) ───────────────────
    rf_best = rf_baseline  # fallback se tuning desabilitado
    if run_tuning:
        _step(11, "Hyperparameter Tuning RF (RandomizedSearchCV no treino)")
        tuner   = TriclassTuner()
        rf_best = tuner.fit(X_train_bal, y_train_bal)
    else:
        _step(11, "Tuning ignorado (run_tuning=False)")

    # ── Etapa 12: Avaliação final no test_set (UMA ÚNICA VEZ) ─────────────────
    _step(12, "Avaliação final no test_set (UMA ÚNICA VEZ)")
    evaluator = TriclassEvaluator(save_plots=save_plots)

    result_rf_base = evaluator.evaluate(
        rf_baseline, X_test_vt.values, y_test.values,
        label="RF Baseline",
    )
    result_rf_best = evaluator.evaluate(
        rf_best, X_test_vt.values, y_test.values,
        label="RF Otimizado" if run_tuning else "RF Baseline",
    )
    result_mlp = evaluator.evaluate(
        mlp_pipe, X_test_vt.values, y_test.values,
        label="MLP Pipeline",
    )
    evaluator.compare(result_rf_base, result_rf_best, result_mlp)

    # ── Etapa 13: Validação semântica BOTNET ──────────────────────────────────
    _step(13, "Validação semântica BOTNET (ground truth)")
    semantic_validator = BotnetSemanticValidator(
        data_original=data_for_validation,
        test_indices=test_original_indices,
    )
    botnet_result = semantic_validator.validate(rf_best, X_test_vt.values)

    # ── Etapa 14: Importância de features + salvamento ────────────────────────
    _step(14, "Importância de features + salvamento de artefatos")

    # 14a — Impurity-based importance (Gini, nativa do RF)
    _plot_feature_importance(
        rf_best, selected_features, computed_feats, save_plots
    )

    # 14b — Permutation Importance no test set
    #        Razão: importância Gini superestima features de alta cardinalidade
    #        (ex: porta, protocolo). A permutation importance no test set mede o
    #        impacto real de cada feature na métrica de generalização.
    #        Localização obrigatória: APÓS Etapa 12 (avaliação já realizada);
    #        uso do test set é apenas diagnóstico — não influencia nenhum modelo.
    print(f"\n[14b] Permutation Importance no test set "
          f"({PERMUTATION_N_REPEATS} repetições)...")
    perm_result = _run_permutation_importance(
        rf_best,
        X_test_vt.values,
        y_test.values,
        selected_features,
        computed_feats,
        n_repeats=PERMUTATION_N_REPEATS,
        save=save_plots,
    )

    # 14c — Ablation Study: retreinar sem features de identidade
    #        Mede quanto do F1 é explicado por atalhos de identidade
    #        (Dst Port, Src Port, Protocol, etc.) em vez de padrões comportamentais.
    print(f"\n[14c] Ablation Study — retreinar sem features de identidade...")
    ablation_result = _run_ablation_study(
        X_train_bal,
        y_train_bal,
        X_test_vt,
        y_test.values,
        selected_features,
        result_rf_best.f1_macro,
        save=save_plots,
    )

    # Salvar todos os artefatos
    joblib.dump(rf_best,           MODELS_TRICLASS_DIR / "rf_triclass.joblib")
    joblib.dump(mlp_pipe,          MODELS_TRICLASS_DIR / "mlp_triclass.joblib")
    joblib.dump(imputer,           MODELS_TRICLASS_DIR / "imputer.joblib")
    joblib.dump(vt,                MODELS_TRICLASS_DIR / "variance_filter.joblib")
    joblib.dump(selected_features, MODELS_TRICLASS_DIR / "selected_features.joblib")
    joblib.dump(computed_feats,    MODELS_TRICLASS_DIR / "computed_features.joblib")

    print(f"\n  Artefatos salvos em {MODELS_TRICLASS_DIR}/")

    # ── Resumo final ───────────────────────────────────────────────────────────
    _banner("Pipeline concluído com sucesso!")
    print(f"  Modelos  : {MODELS_TRICLASS_DIR}/")
    print(f"  Plots    : {OUTPUTS_TRICLASS}/")
    print()
    print(f"  RF Best  — F1 Macro: {result_rf_best.f1_macro:.4f} | "
          f"MCC: {result_rf_best.mcc:.4f} | "
          f"Recall Cl2: {result_rf_best.recall_class2:.4f}")
    print(f"  MLP Pipe — F1 Macro: {result_mlp.f1_macro:.4f} | "
          f"MCC: {result_mlp.mcc:.4f} | "
          f"Recall Cl2: {result_mlp.recall_class2:.4f}")
    print(f"  BOTNET recall semântico: {botnet_result.recall:.4f} "
          f"({'PASSOU' if botnet_result.passed else 'FALHOU'})")
    print(f"  Ablation F1 (sem identidade): {ablation_result['f1_ablation']:.4f} "
          f"| Queda: {ablation_result['f1_drop']:.4f} "
          f"({'OK — comportamental' if ablation_result['f1_drop'] < 0.05 else 'ALERTA — atalho de identidade'})")

    return {
        "rf_baseline":         result_rf_base,
        "rf_best":             result_rf_best,
        "mlp":                 result_mlp,
        "botnet_validation":   botnet_result,
        "cv_rf":               cv_rf,
        "cv_mlp":              cv_mlp,
        "selected_features":   selected_features,
        "permutation_importance": perm_result,
        "ablation":            ablation_result,
    }


# ── Helpers internos ───────────────────────────────────────────────────────────

def _banner(msg: str) -> None:
    print("\n" + "=" * 65)
    print(f"  {msg}")
    print("=" * 65)


def _step(n: int, msg: str) -> None:
    print(f"\n[{n}/14] {msg}...")


def _print_dist(label: str, y: pd.Series) -> None:
    vc = y.value_counts().sort_index()
    parts = [
        f"Cl{cls}({CLASS_NAMES.get(cls,'?')})={cnt:,} ({100*cnt/len(y):.1f}%)"
        for cls, cnt in vc.items()
    ]
    print(f"  {label}: {' | '.join(parts)}")


def _run_permutation_importance(
    rf,
    X_test: np.ndarray,
    y_test: np.ndarray,
    feature_names: list[str],
    computed_feats: list[str],
    n_repeats: int = PERMUTATION_N_REPEATS,
    save: bool = True,
) -> dict:
    """
    Permutation Importance no test set — diagnóstico de atalhos de identidade.

    Por que usar o test set aqui (e não o treino):
      - O test set representa dados "do mundo real" nunca vistos.
      - Permutação no treino mede apenas quais features o modelo memorizou.
      - Permutação no teste mede quais features contribuem para generalização.
      - Uso diagnóstico: NÃO altera nenhum modelo; é análise post-hoc.

    Como interpretar:
      - Importância alta (média >> 0) → feature genuinamente útil.
      - Importância ≈ 0              → feature irrelevante para generalização.
      - Importância negativa         → feature introduz ruído (raramente acontece).
      - Features de identidade com alta importância → risco de overfitting
        para características do dataset, não do problema real.

    Returns
    -------
    dict com 'importances_mean', 'importances_std', 'sorted_df', 'identity_warning'
    """
    from sklearn.inspection import permutation_importance

    print(f"  Calculando permutation importance ({n_repeats} repetições)...")
    result = permutation_importance(
        rf, X_test, y_test,
        n_repeats=n_repeats,
        scoring="f1_macro",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    df_perm = pd.DataFrame({
        "feature":    feature_names,
        "mean":       result.importances_mean,
        "std":        result.importances_std,
    }).sort_values("mean", ascending=False)

    df_perm["identity"] = df_perm["feature"].isin(IDENTITY_FEATURES)
    df_perm["nova"]     = df_perm["feature"].isin(computed_feats)

    # Diagnóstico: features de identidade no top-10?
    top10 = df_perm.head(10)
    identity_in_top10 = top10[top10["identity"]]["feature"].tolist()
    identity_warning  = len(identity_in_top10) > 0

    print(f"\n  Top 15 features (permutation importance — test set):")
    print(f"  {'Feature':<35} {'Mean':>8} {'Std':>8} {'Tipo'}")
    print(f"  {'-'*65}")
    for _, row in df_perm.head(15).iterrows():
        tipo = "IDENTIDADE" if row["identity"] else ("NOVA" if row["nova"] else "original")
        print(f"  {row['feature']:<35} {row['mean']:>8.5f} {row['std']:>8.5f}  {tipo}")

    if identity_warning:
        print(f"\n  ⚠ ALERTA: features de identidade no Top-10 de generalização:")
        for f in identity_in_top10:
            row = df_perm[df_perm["feature"] == f].iloc[0]
            print(f"    {f}: mean={row['mean']:.5f} ± {row['std']:.5f}")
        print("  → Considerar ablation study (sub-passo 14c) para quantificar o impacto.")
    else:
        print("\n  ✓ Nenhuma feature de identidade no Top-10 — padrão comportamental dominante.")

    if save:
        _plot_permutation_importance(df_perm)

    return {
        "importances_mean": result.importances_mean,
        "importances_std":  result.importances_std,
        "sorted_df":        df_perm,
        "identity_warning": identity_warning,
        "identity_in_top10": identity_in_top10,
    }


def _plot_permutation_importance(df_perm: pd.DataFrame) -> None:
    """Barras horizontais com intervalo de confiança por feature."""
    try:
        import matplotlib.pyplot as plt
        from matplotlib.patches import Patch

        top = df_perm.head(25).iloc[::-1]  # ordem crescente para barh
        colors = [
            "crimson"    if r["identity"] else
            "darkorange" if r["nova"]     else
            "steelblue"
            for _, r in top.iterrows()
        ]

        fig, ax = plt.subplots(figsize=(10, 9))
        ax.barh(top["feature"], top["mean"], xerr=top["std"],
                color=colors, alpha=0.85, capsize=3, ecolor="gray")
        ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
        ax.set_xlabel("Redução de F1 Macro ao permutar (média ± std)")
        ax.set_title(
            "Permutation Importance — RF Triclasse (test set)\n"
            "Crimson = identidade  |  Laranja = comportamental nova  |  Azul = original"
        )
        ax.legend(handles=[
            Patch(color="crimson",    label="Feature de identidade (suspeita)"),
            Patch(color="darkorange", label="Feature comportamental (nova)"),
            Patch(color="steelblue",  label="Feature original InSDN"),
        ])
        plt.tight_layout()
        path = OUTPUTS_TRICLASS / "permutation_importance.png"
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"  Permutation importance salva → {path.name}")
    except Exception as e:
        print(f"  Erro ao plotar permutation importance: {e}")


def _run_ablation_study(
    X_train_bal: np.ndarray,
    y_train_bal: np.ndarray,
    X_test_vt: pd.DataFrame,
    y_test: np.ndarray,
    selected_features: list[str],
    f1_full: float,
    save: bool = True,
) -> dict:
    """
    Ablation Study: retreinar RF sem as features de identidade.

    Mede quanto do F1 é explicado por atalhos de identidade vs. padrões
    comportamentais reais. Se a queda for pequena (< 0.05), o modelo
    generaliza por comportamento — não por memorizar porta/protocolo.

    Por que retreinar e não apenas mascarar:
      - Mascarar colunas no teste com modelo treinado no set completo não é
        ablation real — o modelo já aprendeu relações com essas features.
      - Retreinar sem elas força o modelo a aprender sem esses sinais.

    Por que usar X_train_bal (pós-SMOTE) e não X_train_vt (pré-SMOTE):
      - O modelo de referência foi treinado em X_train_bal.
      - Para comparação justa, o modelo ablation deve ter as mesmas condições
        exceto pela ausência das features de identidade.

    Returns
    -------
    dict com 'f1_full', 'f1_ablation', 'f1_drop', 'features_removed',
             'features_kept', 'identity_warning'
    """
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import f1_score

    # Identificar quais features de identidade estão presentes após VT
    identity_present = [f for f in IDENTITY_FEATURES if f in selected_features]
    features_kept    = [f for f in selected_features if f not in IDENTITY_FEATURES]

    if not identity_present:
        print("  Nenhuma feature de identidade presente após VarianceThreshold — ablation ignorada.")
        return {
            "f1_full":          f1_full,
            "f1_ablation":      f1_full,
            "f1_drop":          0.0,
            "features_removed": [],
            "features_kept":    features_kept,
            "identity_warning": False,
        }

    print(f"  Features de identidade a remover: {identity_present}")
    print(f"  Features restantes no ablation: {len(features_kept)}")

    # Índices das colunas a manter em X_train_bal (numpy array)
    keep_idx = [selected_features.index(f) for f in features_kept]

    X_train_abl = X_train_bal[:, keep_idx]
    X_test_abl  = X_test_vt[features_kept].values

    rf_ablation = RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    rf_ablation.fit(X_train_abl, y_train_bal)
    y_pred_abl  = rf_ablation.predict(X_test_abl)
    f1_ablation = float(f1_score(y_test, y_pred_abl, average="macro", zero_division=0))
    f1_drop     = f1_full - f1_ablation

    print(f"\n  F1 Macro com todas as features    : {f1_full:.4f}")
    print(f"  F1 Macro sem features identidade  : {f1_ablation:.4f}")
    print(f"  Queda                             : {f1_drop:+.4f}")

    if f1_drop < 0.02:
        verdict = "✓ Queda < 0.02 — modelo generaliza por comportamento, não por identidade."
        warning = False
    elif f1_drop < 0.05:
        verdict = "⚠ Queda entre 0.02–0.05 — features de identidade contribuem moderadamente."
        warning = True
    else:
        verdict = "✗ Queda > 0.05 — modelo depende fortemente de atalhos de identidade."
        warning = True
    print(f"\n  Veredito: {verdict}")

    if save:
        _plot_ablation(f1_full, f1_ablation, identity_present, features_kept)

    return {
        "f1_full":          f1_full,
        "f1_ablation":      f1_ablation,
        "f1_drop":          f1_drop,
        "features_removed": identity_present,
        "features_kept":    features_kept,
        "identity_warning": warning,
    }


def _plot_ablation(
    f1_full: float,
    f1_ablation: float,
    features_removed: list[str],
    features_kept: list[str],
) -> None:
    """Gráfico de barras comparando F1 completo vs. ablation."""
    try:
        import matplotlib.pyplot as plt

        labels = ["Modelo completo", "Sem features\nde identidade"]
        values = [f1_full, f1_ablation]
        colors = ["steelblue", "darkorange"]

        fig, ax = plt.subplots(figsize=(7, 5))
        bars = ax.bar(labels, values, color=colors, alpha=0.85, width=0.4)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.003,
                    f"{val:.4f}", ha="center", va="bottom", fontsize=12,
                    fontweight="bold")

        ax.set_ylim(max(0, min(values) - 0.05), min(1.0, max(values) + 0.05))
        ax.set_ylabel("F1 Macro (test set)")
        ax.set_title(
            f"Ablation Study — Features de Identidade\n"
            f"Removidas: {', '.join(features_removed)}"
        )

        drop = f1_full - f1_ablation
        color = "darkgreen" if drop < 0.02 else ("orange" if drop < 0.05 else "crimson")
        ax.text(0.5, 0.05,
                f"Queda: {drop:+.4f}  "
                f"({'OK' if drop < 0.02 else 'MODERADA' if drop < 0.05 else 'ALTA'})",
                transform=ax.transAxes, ha="center", fontsize=11,
                color=color, fontweight="bold")

        ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        path = OUTPUTS_TRICLASS / "ablation_identity_features.png"
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"  Ablation plot salvo → {path.name}")
    except Exception as e:
        print(f"  Erro ao plotar ablation: {e}")


def _plot_feature_importance(
    rf,
    selected_features: list[str],
    computed_feats: list[str],
    save: bool,
) -> None:
    try:
        import matplotlib.pyplot as plt
        from matplotlib.patches import Patch

        df_imp = pd.DataFrame({
            "feature":    selected_features,
            "importance": rf.feature_importances_,
        }).sort_values("importance", ascending=False).head(25)

        df_imp["nova"] = df_imp["feature"].isin(computed_feats)

        fig, ax = plt.subplots(figsize=(10, 8))
        cores = ["darkorange" if n else "steelblue"
                 for n in df_imp["nova"].iloc[::-1]]
        ax.barh(df_imp["feature"].iloc[::-1],
                df_imp["importance"].iloc[::-1],
                color=cores)
        ax.set_xlabel("Importância (redução de impureza)")
        ax.set_title("Top 25 Features — RF Triclasse\nLaranja = features comportamentais")
        ax.legend(handles=[
            Patch(color="darkorange", label="Feature comportamental (nova)"),
            Patch(color="steelblue",  label="Feature original InSDN"),
        ])
        plt.tight_layout()

        if save:
            path = OUTPUTS_TRICLASS / "feature_importance.png"
            plt.savefig(path, dpi=150, bbox_inches="tight")
            print(f"  Importância de features salva → {path.name}")
        plt.close()
    except Exception as e:
        print(f"  Não foi possível plotar importância: {e}")


# ── CLI ────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(
        description="Pipeline Triclasse — Detecção de DDoS em SDN (InSDN v4)"
    )
    parser.add_argument(
        "--no-tuning", action="store_true",
        help="Pular hyperparameter tuning (mais rápido)"
    )
    parser.add_argument(
        "--no-eda", action="store_true",
        help="Pular análise exploratória"
    )
    parser.add_argument(
        "--no-plots", action="store_true",
        help="Não salvar gráficos"
    )
    parser.add_argument(
        "--run-id", type=str, default=None,
        help="Identificador da run (ex: 'experimento_v1')"
    )
    args = parser.parse_args()

    run_triclass_pipeline(
        run_tuning=not args.no_tuning,
        run_eda=not args.no_eda,
        save_plots=not args.no_plots,
        run_id=args.run_id,
    )


### Execução do Pipeline Triclasse
Descomente a célula abaixo para executar:


In [ ]:
# run_triclass_pipeline(run_tuning=False, run_eda=True, save_plots=True, run_id='notebook_triclass')